# Prediksi Tinggi Muka Air Sungai — RMSE 0.81 Target Challenger

Full pipeline Kaggle: audit, feature engineering, annual-memory features, rolling-origin validation, heterogeneous ensemble, OOF stacking, final fit, dan submission.


## 1. Import, path, dan konfigurasi model


In [ ]:
"""Training and submission pipeline for river water-level (TMA) forecasting.

The pipeline is deliberately exogenous and non-recursive: every feature used for a
future prediction is available in ``data_lingkungan.csv`` or is static/calendar
information.  This avoids compounding autoregressive error over the 242-day test
horizon.

v2 additions (senior review follow-up, 2026-07-15):
- Validation now scores the FULL ensemble per fold instead of a single seed, so the
  reported metric matches what actually goes into the submission.
- The ensemble is diversified across CatBoost (varying depth/lr/l2) and LightGBM
  (varying leaves/feature-subsampling) members instead of 3 identical CatBoost seeds.
- Spike-downweighting and feature-pruning are togglable so they can be ablated
  instead of trusted on intuition alone.
- Optional lightweight hyperparameter sweep and optional CatBoost quantile model.

Training is never started implicitly. The execution cell defaults to audit mode.
"""

from __future__ import annotations

import importlib.metadata
import json
import logging
import math
import platform
import re
import sys
import traceback
import zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


LOGGER = logging.getLogger("tma_forecasting")
KEYS = ["datetime", "nama_pos"]
ID_PATTERN = re.compile(
    r"^(?P<datetime>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (?P<nama_pos>.+)$"
)


@dataclass(frozen=True)
class Paths:
    root: Path = Path("datasets")
    artifacts: Path = Path("artifacts")

    @property
    def train(self) -> Path:
        return self.root / "train.csv"

    @property
    def test(self) -> Path:
        return self.root / "test.csv"

    @property
    def sample_submission(self) -> Path:
        return self.root / "sample_submission.csv"

    @property
    def coordinates(self) -> Path:
        return self.root / "data_pendukung" / "koordinat_pos.csv"

    @property
    def environment(self) -> Path:
        return self.root / "data_pendukung" / "data_lingkungan.csv"

    @property
    def rivers(self) -> Path:
        return (
            self.root
            / "data_pendukung"
            / "HydroRIVERS_v10_au_shp"
            / "HydroRIVERS_v10_au.shp"
        )


@dataclass(frozen=True)
class EnsembleMemberConfig:
    """One ensemble member.

    Members are intentionally heterogeneous (family / depth / leaves / feature
    subsampling) so the ensemble reduces *structural* bias, not just seed variance.
    Every member still predicts the same robust-normalized per-station target, so
    predictions stay directly averageable on the z-scale regardless of family.
    """

    name: str
    family: str  # "catboost" or "lightgbm"
    seed: int
    # CatBoost-specific
    depth: int = 8
    learning_rate: float = 0.035
    l2_leaf_reg: float = 10.0
    # LightGBM-specific
    num_leaves: int = 63
    feature_fraction: float = 0.9
    bagging_fraction: float = 0.85
    min_child_samples: int = 25


DEFAULT_ENSEMBLE: tuple[EnsembleMemberConfig, ...] = (
    EnsembleMemberConfig("cb_shallow", "catboost", seed=17, depth=6, learning_rate=0.045, l2_leaf_reg=8.0),
    EnsembleMemberConfig("cb_base", "catboost", seed=41, depth=8, learning_rate=0.02, l2_leaf_reg=10.0),
    EnsembleMemberConfig("cb_deep", "catboost", seed=83, depth=8, learning_rate=0.025, l2_leaf_reg=14.0),
    EnsembleMemberConfig("lgbm_a", "lightgbm", seed=7, num_leaves=63, learning_rate=0.03,
                          feature_fraction=0.85, bagging_fraction=0.8),
    EnsembleMemberConfig("lgbm_b", "lightgbm", seed=29, num_leaves=127, learning_rate=0.02,
                          feature_fraction=0.7, bagging_fraction=0.8),
)


@dataclass(frozen=True)
class ModelConfig:
    ensemble: tuple[EnsembleMemberConfig, ...] = DEFAULT_ENSEMBLE
    default_iterations: int = 1800
    huber_delta: float = 1.5
    early_stopping_rounds: int = 200
    verbose_every: int = 100
    upstream_decay_km: float = 75.0
    snapshot_interval_seconds: int = 300
    # --- Ablation / optimization toggles (senior review points 1-4) ---
    use_spike_weights: bool = True
    prune_features: bool = False
    feature_importance_source: str | None = None  # path to a prior feature_importance.csv
    prune_keep_fraction: float = 0.85
    enable_quantile_model: bool = False
    quantile_levels: tuple[float, ...] = (0.1, 0.5, 0.9)
    # --- Per-station harmonic baseline feature (Wonogiri Dam follow-up) ---
    enable_harmonic_feature: bool = True
    harmonic_ridge_alpha: float = 5.0
    # --- Recency/persistence anchor feature (station-aware stacking follow-up) ---
    enable_anchor_feature: bool = True
    anchor_recent_window_days: float = 30.0
    anchor_tau_days: float = 365.0


FOLDS = (
    ("sep_2023", "2023-09-18 18:00:00", "2023-09-19 06:00:00", "2024-05-18 18:00:00"),
    ("may_2024", "2024-05-18 18:00:00", "2024-05-19 06:00:00", "2025-01-18 18:00:00"),
    ("sep_2024", "2024-09-18 18:00:00", "2024-09-19 06:00:00", "2025-05-18 18:00:00"),
    ("jan_2025", "2025-01-18 18:00:00", "2025-01-19 06:00:00", "2025-09-18 18:00:00"),
)


RIVER_NUMERIC = [
    "LENGTH_KM",
    "DIST_DN_KM",
    "DIST_UP_KM",
    "CATCH_SKM",
    "UPLAND_SKM",
    "DIS_AV_CMS",
    "ORD_STRA",
    "ORD_CLAS",
    "ORD_FLOW",
]

RAW_STATE_COLUMNS = [
    "soil_moisture_0_7cm",
    "soil_moisture_7_28cm",
    "soil_moisture_28_100cm",
    "soil_moisture_100_255cm",
    "surface_pressure_hpa",
    "pressure_msl_hpa",
    "rmm1",
    "rmm2",
    "mjo_phase",
    "mjo_amplitude",
    "mjo_active",
    "nino_34",
]

CATEGORICAL_COLUMNS = [
    "nama_pos",
    "landcover_name",
    "hour_cat",
    "month_cat",
    "mjo_phase_cat",
    "main_riv_cat",
    "hybas_l12_cat",
    "landcover_class_cat",
    "endorheic_cat",
]

NON_FEATURE_COLUMNS = {
    "datetime",
    "id",
    "tma_mdpl",
    "_row_order",
    "mjo_phase",
    "landcover_class",
    "HYRIV_ID",
    "NEXT_DOWN",
    "MAIN_RIV",
    "HYBAS_L12",
    "ENDORHEIC",
    "wind_direction_deg",
}


## 2. Parsing ID, pemuatan data, dan audit schema

Missing baris target tidak dibuat menjadi pseudo-label. Semua key, station set, jam, dan urutan sample submission diperiksa sebelum feature engineering.


In [ ]:
def parse_submission_ids(frame: pd.DataFrame) -> pd.DataFrame:
    """Parse IDs without splitting station names that themselves contain ' - '."""
    parsed = frame["id"].str.extract(ID_PATTERN)
    if parsed.isna().any(axis=None):
        bad = frame.loc[parsed.isna().any(axis=1), "id"].head(3).tolist()
        raise ValueError(f"Invalid submission IDs; examples: {bad}")
    result = frame.copy()
    result["datetime"] = pd.to_datetime(parsed["datetime"], errors="raise")
    result["nama_pos"] = parsed["nama_pos"]
    result["_row_order"] = np.arange(len(result), dtype=np.int64)
    return result


def load_inputs(paths: Paths) -> tuple[pd.DataFrame, ...]:
    train = pd.read_csv(paths.train, parse_dates=["datetime"])
    test_raw = pd.read_csv(paths.test)
    sample = pd.read_csv(paths.sample_submission)
    coordinates = pd.read_csv(paths.coordinates)
    environment = pd.read_csv(paths.environment, parse_dates=["datetime"])
    test = parse_submission_ids(test_raw)
    return train, test, sample, coordinates, environment


def audit_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
) -> dict:
    """Fail early on silent schema, key, time, or station-set changes."""
    expected_train = {"datetime", "nama_pos", "tma_mdpl"}
    if set(train.columns) != expected_train:
        raise ValueError(f"Unexpected train schema: {train.columns.tolist()}")
    if train.duplicated(KEYS).any() or environment.duplicated(KEYS).any():
        raise ValueError("Duplicate (datetime, nama_pos) keys detected")
    if train["tma_mdpl"].isna().any():
        raise ValueError("Target contains explicit NaN values")
    if test["id"].duplicated().any():
        raise ValueError("Duplicate test IDs detected")
    if not test["id"].reset_index(drop=True).equals(sample["id"].reset_index(drop=True)):
        raise ValueError("test.csv and sample_submission.csv IDs/order differ")

    train_stations = set(train["nama_pos"])
    station_sets = {
        "test": set(test["nama_pos"]),
        "coordinates": set(coordinates["nama_pos"]),
        "environment": set(environment["nama_pos"]),
    }
    for label, stations in station_sets.items():
        if stations != train_stations:
            raise ValueError(
                f"Station mismatch for {label}: missing={sorted(train_stations-stations)}, "
                f"extra={sorted(stations-train_stations)}"
            )

    expected_hours = {6, 12, 18}
    if set(train["datetime"].dt.hour) - expected_hours:
        raise ValueError("Train contains target hours outside 06, 12, 18")
    if set(test["datetime"].dt.hour) != expected_hours:
        raise ValueError("Test hours are not exactly 06, 12, 18")

    env_counts = environment.groupby("nama_pos", observed=True).size()
    expected_env_rows = len(
        pd.date_range(environment["datetime"].min(), environment["datetime"].max(), freq="h")
    )
    if not (env_counts == expected_env_rows).all():
        raise ValueError("Environment is not a complete hourly grid for every station")

    return {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "environment_rows": int(len(environment)),
        "stations": int(len(train_stations)),
        "train_start": str(train["datetime"].min()),
        "train_end": str(train["datetime"].max()),
        "test_start": str(test["datetime"].min()),
        "test_end": str(test["datetime"].max()),
        "environment_missing_cells": int(environment.isna().sum().sum()),
        "solar_minus_999_cells": int((environment["solar_radiation_mj_m2"] <= -900).sum()),
    }


## 3. HydroRIVERS dan graf upstream

Koordinat pos dicocokkan dalam UTM 49S. Traversal `NEXT_DOWN` membentuk bobot lingkungan upstream yang menurun terhadap jarak hidraulik.


In [ ]:
def _read_local_rivers(path: Path, coordinates: pd.DataFrame) -> gpd.GeoDataFrame:
    pad = 0.75
    bbox = (
        coordinates["longitude"].min() - pad,
        coordinates["latitude"].min() - pad,
        coordinates["longitude"].max() + pad,
        coordinates["latitude"].max() + pad,
    )
    rivers = gpd.read_file(path, bbox=bbox)
    required = {
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
        "geometry",
    }
    missing = required - set(rivers.columns)
    if missing:
        raise ValueError(f"HydroRIVERS is missing columns: {sorted(missing)}")
    if rivers.crs is None:
        raise ValueError("HydroRIVERS has no CRS")
    return rivers


def match_stations_to_rivers(
    coordinates: pd.DataFrame, rivers: gpd.GeoDataFrame
) -> pd.DataFrame:
    points = gpd.GeoDataFrame(
        coordinates.copy(),
        geometry=gpd.points_from_xy(coordinates["longitude"], coordinates["latitude"]),
        crs="EPSG:4326",
    )
    # All supplied gauges are inside UTM zone 49S (108E-114E).
    points_utm = points.to_crs("EPSG:32749")
    rivers_utm = rivers.to_crs("EPSG:32749")
    nearest = gpd.sjoin_nearest(
        points_utm,
        rivers_utm,
        how="left",
        distance_col="river_distance_m",
    )
    nearest = (
        nearest.sort_values("river_distance_m")
        .loc[lambda x: ~x.index.duplicated(keep="first")]
        .sort_index()
    )
    if nearest["HYRIV_ID"].isna().any():
        raise ValueError("At least one station could not be matched to HydroRIVERS")

    keep = [
        "nama_pos",
        "latitude",
        "longitude",
        "river_distance_m",
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
    ]
    static = pd.DataFrame(nearest[keep]).reset_index(drop=True)
    static["river_match_confidence"] = np.exp(-static["river_distance_m"] / 1000.0)
    for col in ["CATCH_SKM", "UPLAND_SKM", "DIS_AV_CMS", "LENGTH_KM"]:
        static[f"log1p_{col.lower()}"] = np.log1p(static[col].clip(lower=0))
    static["main_riv_cat"] = static["MAIN_RIV"].astype("Int64").astype(str)
    static["hybas_l12_cat"] = static["HYBAS_L12"].astype("Int64").astype(str)
    static["endorheic_cat"] = static["ENDORHEIC"].astype("Int64").astype(str)
    return static


def build_upstream_weights(
    static: pd.DataFrame,
    rivers: gpd.GeoDataFrame,
    decay_km: float,
) -> pd.DataFrame:
    """Create target-by-source weights from exact NEXT_DOWN reach traversal."""
    stations = static["nama_pos"].tolist()
    reach_by_station = dict(zip(static["nama_pos"], static["HYRIV_ID"].astype(int)))
    dist_by_station = dict(zip(static["nama_pos"], static["DIST_DN_KM"].astype(float)))
    station_by_reach = {reach: station for station, reach in reach_by_station.items()}
    next_down = dict(zip(rivers["HYRIV_ID"].astype(int), rivers["NEXT_DOWN"].astype(int)))

    weights = pd.DataFrame(0.0, index=stations, columns=stations)
    for source in stations:
        weights.loc[source, source] = 1.0
        current = reach_by_station[source]
        seen = {current}
        for _ in range(len(next_down) + 1):
            current = next_down.get(current, 0)
            if current == 0 or current in seen:
                break
            seen.add(current)
            target = station_by_reach.get(current)
            if target is not None:
                distance = max(0.0, dist_by_station[source] - dist_by_station[target])
                weights.loc[target, source] = max(
                    weights.loc[target, source], math.exp(-distance / decay_km)
                )
    return weights.div(weights.sum(axis=1), axis=0)


def _add_upstream_environment(
    environment: pd.DataFrame,
    weights: pd.DataFrame,
    source_columns: Iterable[str],
) -> pd.DataFrame:
    result = environment
    station_order = weights.columns.tolist()
    for column in source_columns:
        pivot = result.pivot(index="datetime", columns="nama_pos", values=column).reindex(
            columns=station_order
        )
        upstream = pd.DataFrame(
            pivot.to_numpy(dtype=float) @ weights.to_numpy(dtype=float).T,
            index=pivot.index,
            columns=weights.index,
        )
        long = (
            upstream.rename_axis(columns="nama_pos")
            .stack(future_stack=True)
            .rename(f"upstream_{column}")
            .reset_index()
        )
        result = result.merge(long, on=KEYS, how="left", validate="one_to_one")
    return result


## 4. Sanitasi lingkungan dan fitur antecedent

Menangani sentinel radiasi `-999`, missing tail, arah angin siklik, hujan/soil upstream, rolling window, dan antecedent precipitation index secara kausal.


In [ ]:
def prepare_environment(
    environment: pd.DataFrame, upstream_weights: pd.DataFrame
) -> pd.DataFrame:
    env = environment.sort_values(KEYS).copy()

    # These columns are byte-identical in the supplied data; retain one copy only.
    if not env["rainfall_mm"].equals(env["rainfall_openmeteo_mm"]):
        raise ValueError("Rainfall duplicate columns have unexpectedly diverged")
    env = env.drop(columns="rainfall_openmeteo_mm")

    # -999 is an undocumented sentinel, not physically possible solar radiation.
    env["solar_radiation_missing"] = (
        env["solar_radiation_mj_m2"].isna()
        | (env["solar_radiation_mj_m2"] <= -900)
    ).astype("int8")
    env.loc[env["solar_radiation_mj_m2"] <= -900, "solar_radiation_mj_m2"] = np.nan

    # Missing indicators are retained even after causal forward filling.
    for col in RAW_STATE_COLUMNS:
        env[f"{col}_missing"] = env[col].isna().astype("int8")

    by_station = env.groupby("nama_pos", sort=False, observed=True)
    short_state = [c for c in RAW_STATE_COLUMNS if c != "nino_34"]
    env[short_state] = by_station[short_state].ffill(limit=24)
    # Nino 3.4 is a slowly varying monthly index; a 31-day causal hold is appropriate.
    env[["nino_34"]] = by_station[["nino_34"]].ffill(limit=31 * 24)

    radians = np.deg2rad(env["wind_direction_deg"])
    env["wind_direction_sin"] = np.sin(radians)
    env["wind_direction_cos"] = np.cos(radians)
    env["dewpoint_depression_c"] = env["temperature_c"] - env["dew_point_c"]

    env = _add_upstream_environment(
        env,
        upstream_weights,
        ["rainfall_mm", "soil_moisture_0_7cm"],
    )
    env = env.sort_values(KEYS).reset_index(drop=True)
    grouped = env.groupby("nama_pos", sort=False, observed=True)

    rolling_sum_specs = {
        "rainfall_mm": (6, 24, 72, 168, 336),
        "upstream_rainfall_mm": (6, 24, 72, 168, 336),
    }
    for col, windows in rolling_sum_specs.items():
        for hours in windows:
            env[f"{col}_sum_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(w, min_periods=max(1, w // 4)).sum()
            )
        env[f"{col}_max_24h"] = grouped[col].transform(
            lambda s: s.rolling(24, min_periods=6).max()
        )
        for half_life in (12, 48, 168):
            env[f"{col}_api_halflife_{half_life}h"] = grouped[col].transform(
                lambda s, h=half_life: s.ewm(
                    halflife=h, adjust=False, min_periods=1
                ).mean()
            )

    rolling_mean_specs = {
        "soil_moisture_0_7cm": (24, 168),
        "soil_moisture_28_100cm": (24, 168),
        "upstream_soil_moisture_0_7cm": (24, 168),
        "temperature_c": (24, 72),
        "humidity_pct": (24, 72),
        "surface_pressure_hpa": (24,),
    }
    for col, windows in rolling_mean_specs.items():
        for hours in windows:
            env[f"{col}_mean_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(
                    w, min_periods=max(1, w // 4)
                ).mean()
            )

    # Physical sanitation. Values are not fitted against the future distribution.
    env["rainfall_mm"] = env["rainfall_mm"].clip(lower=0)
    env["rainfall_mm_max_24h"] = env["rainfall_mm_max_24h"].clip(lower=0)
    env["solar_radiation_mj_m2"] = env["solar_radiation_mj_m2"].clip(lower=0)
    for col in [c for c in env.columns if "soil_moisture" in c]:
        env[col] = env[col].clip(0, 1)
    return env


## 5. Kalender, kategori, matriks fitur, dan feature pruning

`apply_feature_pruning` di sini murni opt-in: kalau `PRUNE_FEATURES=False` atau sumber `feature_importance.csv` belum ada, seluruh fitur tetap dipakai. Baru berguna setelah minimal satu `MODE="validate"` dijalankan.


In [ ]:
def add_calendar_and_categories(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    dt = out["datetime"]
    doy = dt.dt.dayofyear.astype(float)
    hour = dt.dt.hour.astype(float)
    out["dayofyear_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["dayofyear_cos"] = np.cos(2 * np.pi * doy / 365.25)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["is_wet_season"] = dt.dt.month.isin([11, 12, 1, 2, 3, 4]).astype("int8")
    out["hour_cat"] = dt.dt.hour.astype(str)
    out["month_cat"] = dt.dt.month.astype(str)
    out["mjo_phase_cat"] = out["mjo_phase"].round().astype("Int64").astype(str)
    out["landcover_class_cat"] = out["landcover_class"].astype("Int64").astype(str)
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna("__MISSING__").astype(str)
    return out


def build_feature_tables(
    train: pd.DataFrame,
    test: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
    paths: Paths,
    config: ModelConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rivers = _read_local_rivers(paths.rivers, coordinates)
    static = match_stations_to_rivers(coordinates, rivers)
    weights = build_upstream_weights(static, rivers, config.upstream_decay_km)
    env_features = prepare_environment(environment, weights)

    train_features = train.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    test_features = test.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    train_features = train_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    test_features = test_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    if (
        train_features["rainfall_mm"].isna().any()
        or test_features["rainfall_mm"].isna().any()
    ):
        raise ValueError("Some target timestamps have no matching environmental row")

    train_features = add_calendar_and_categories(train_features)
    test_features = add_calendar_and_categories(test_features)

    if config.enable_harmonic_feature:
        # Placeholder only. The real, leakage-safe values are computed per-fold
        # (fit on the train prefix only, applied to fit+valid) inside
        # run_validation / fit_final_and_submit -- never here, since here we
        # don't yet know which rows are "future" relative to a given cutoff.
        train_features["harmonic_baseline_mdpl"] = np.nan
        test_features["harmonic_baseline_mdpl"] = np.nan

    if config.enable_anchor_feature:
        # Same placeholder pattern: horizon_days is cutoff-relative (so it can
        # only be computed per-fold), and recent_anomaly_mdpl/anchor_basis_mdpl
        # depend on the fold's causal harmonic baseline residual tail.
        train_features["horizon_days"] = np.nan
        train_features["recent_anomaly_mdpl"] = np.nan
        train_features["anchor_basis_mdpl"] = np.nan
        test_features["horizon_days"] = np.nan
        test_features["recent_anomaly_mdpl"] = np.nan
        test_features["anchor_basis_mdpl"] = np.nan

    return train_features, test_features, weights


def feature_columns(frame: pd.DataFrame) -> list[str]:
    cols = [c for c in frame.columns if c not in NON_FEATURE_COLUMNS]
    # Target-derived features are intentionally forbidden.
    forbidden = [c for c in cols if c.startswith("tma_") or "target" in c.lower()]
    if forbidden:
        raise ValueError(f"Target leakage features detected: {forbidden}")
    return cols


def apply_feature_pruning(
    columns: list[str],
    importance_source: str | Path | None,
    keep_fraction: float,
) -> tuple[list[str], dict]:
    """Drop the lowest-importance features using a PRIOR run's feature_importance.csv.

    Senior review point 3: feature_importance.csv used to be computed and never
    used. Pruning is opt-in and safe by default -- if no source is given or the file
    can't be found, the full feature set is returned unchanged and the report says
    so, instead of silently changing behaviour.
    """
    report = {
        "requested": importance_source is not None,
        "applied": False,
        "kept": len(columns),
        "dropped": 0,
        "dropped_features": [],
    }
    if importance_source is None:
        return columns, report
    path = Path(importance_source)
    if not path.exists():
        LOGGER.warning("Feature importance source %s not found; keeping all features", path)
        return columns, report

    importance = pd.read_csv(path)
    # Works for both the per-fold/per-member ("fold","member","feature","importance")
    # and per-final-member ("member","family","feature","importance") schemas
    # produced by this notebook.
    agg = importance.groupby("feature", observed=True)["importance"].mean()
    agg = agg.reindex(columns).fillna(0.0)
    n_keep = max(10, int(math.ceil(len(columns) * keep_fraction)))
    kept = set(agg.sort_values(ascending=False).head(n_keep).index.tolist())
    kept_ordered = [c for c in columns if c in kept]
    dropped = sorted(set(columns) - kept)
    report.update(
        applied=True,
        kept=len(kept_ordered),
        dropped=len(dropped),
        dropped_features=dropped,
    )
    LOGGER.info(
        "Feature pruning: kept %d/%d features (dropped %d)",
        len(kept_ordered), len(columns), len(dropped),
    )
    return kept_ordered, report


## 6. Target robust, spike weighting (togglable), model builder multi-family, dan fitur baseline harmonik per pos

Target dinormalisasi robust per pos. Spike satu-titik hanya diturunkan bobotnya bila kedua tetangganya konsisten; kejadian tinggi berkelanjutan tetap berbobot penuh. `compute_sample_weights` membungkus mekanisme ini dengan toggle `use_spike_weights` agar efeknya bisa diablasi di bagian 8b, bukan sekadar diasumsikan bekerja.

Model builder sekarang mendukung dua keluarga (`catboost` dan `lightgbm`) lewat antarmuka yang sama (`build_model` / `fit_member` / `predict_member`), sehingga ensemble bisa terdiri dari anggota yang benar-benar berbeda strukturnya, bukan cuma beda seed.

**Baseline harmonik per pos (`harmonic_baseline_mdpl`)** ditambahkan sebagai fitur numerik, bukan cuma metrik pembanding. Ridge Fourier per pos di-fit kausal HANYA pada prefix training tiap fold (mirror `fit_target_stats`), lalu dievaluasi ke fit+valid pakai kalender saja -- tidak pernah membaca target baris yang diprediksi. Ini follow-up dari investigasi Wonogiri Dam: dua arsitektur berbeda total (ensemble kita dan challenger XGBoost-residual-harmonik) menunjukkan arah bias yang sama, mengindikasikan sinyal yang hilang (kemungkinan operasi bendungan), bukan masalah model -- fitur ini tidak menutup gap itu, tapi memberi trees kurva musiman/diurnal per pos yang eksplisit dan bisa diekstrapolasi 8 bulan ke depan. Toggle: `config.enable_harmonic_feature`.

**Fitur anchor (`horizon_days`, `recent_anomaly_mdpl`, `anchor_basis_mdpl`)** adalah follow-up dari hasil tim lain (station-aware stacking, leaderboard resmi 1.59656 vs punya kita 1.60232). Inti idenya: anomali per pos relatif ke baseline musiman, diukur tepat di cutoff training, dikalikan peluruhan eksponensial terhadap horizon -- ini langsung menyasar pola Wonogiri Dam (bias yang besarnya berubah tergantung seberapa jauh dari cutoff). Bedanya dengan implementasi tim lain: mereka hand-tune alpha koreksi per pos dari OOF; kita ekspos sebagai FITUR mentah ke CatBoost/LightGBM yang sudah ada, biar trees yang belajar sendiri per pos/wilayah seberapa besar harus percaya sinyal ini -- lebih sedikit parameter yang ditarik manual dari cuma 4 fold. `apply_station_features` fit satu kurva Ridge yang sama dipakai untuk kedua fitur (harmonik + anchor), bukan fit dua kali. Toggle: `config.enable_anchor_feature`.


In [ ]:
def fit_target_stats(train: pd.DataFrame) -> pd.DataFrame:
    grouped = train.groupby("nama_pos", observed=True)["tma_mdpl"]
    stats = grouped.agg(
        center="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
    )
    stats["scale"] = ((stats["q75"] - stats["q25"]) / 1.349).clip(lower=0.05)
    return stats[["center", "scale"]]


def normalize_target(frame: pd.DataFrame, stats: pd.DataFrame) -> np.ndarray:
    center = frame["nama_pos"].map(stats["center"])
    scale = frame["nama_pos"].map(stats["scale"])
    if center.isna().any() or scale.isna().any():
        raise ValueError("Target normalization encountered an unseen station")
    return ((frame["tma_mdpl"] - center) / scale).to_numpy(dtype=float)


def denormalize_prediction(
    prediction: np.ndarray, stations: pd.Series, stats: pd.DataFrame
) -> np.ndarray:
    center = stations.map(stats["center"]).to_numpy(dtype=float)
    scale = stations.map(stats["scale"]).to_numpy(dtype=float)
    return center + scale * np.asarray(prediction, dtype=float)


def isolated_spike_weights(train: pd.DataFrame) -> tuple[np.ndarray, int]:
    """Downweight only one-step spikes whose two neighbors agree with each other."""
    ordered = train.sort_values(KEYS).copy()
    group = ordered.groupby("nama_pos", sort=False, observed=True)
    ordered["prev_y"] = group["tma_mdpl"].shift(1)
    ordered["next_y"] = group["tma_mdpl"].shift(-1)
    ordered["prev_t"] = group["datetime"].shift(1)
    ordered["next_t"] = group["datetime"].shift(-1)

    diff_scale = group["tma_mdpl"].transform(
        lambda s: (s.diff().abs().median() * 1.4826)
    ).clip(lower=0.05)
    station_stats = fit_target_stats(ordered)
    robust_scale = ordered["nama_pos"].map(station_stats["scale"])
    midpoint = (ordered["prev_y"] + ordered["next_y"]) / 2.0
    threshold = np.maximum(12.0 * diff_scale, 8.0 * robust_scale)
    neighbors_agree = (ordered["prev_y"] - ordered["next_y"]).abs() <= np.maximum(
        4.0 * diff_scale, 2.0 * robust_scale
    )
    adjacent = (
        ((ordered["datetime"] - ordered["prev_t"]) <= pd.Timedelta(hours=12))
        & ((ordered["next_t"] - ordered["datetime"]) <= pd.Timedelta(hours=12))
    )
    spike = adjacent & neighbors_agree & ((ordered["tma_mdpl"] - midpoint).abs() > threshold)
    ordered["weight"] = np.where(spike, 0.05, 1.0)
    weights = ordered["weight"].reindex(train.index).to_numpy(dtype=float)
    return weights, int(spike.sum())


def compute_sample_weights(train: pd.DataFrame, use_spike_weights: bool) -> tuple[np.ndarray, int]:
    """Wraps isolated_spike_weights with an ablation on/off switch.

    Senior review point 3: the spike-downweighting thresholds were never validated
    against an actual metric delta. `spike_count` is always returned (for reporting)
    even when the weights themselves are neutralized, so ablation runs stay
    comparable apples-to-apples on everything except the weighting itself.
    """
    weights, spike_count = isolated_spike_weights(train)
    if not use_spike_weights:
        weights = np.ones_like(weights)
    return weights, spike_count


def metrics_frame(y_true: np.ndarray, y_pred: np.ndarray, stations: pd.Series) -> dict:
    errors = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    station_mae = pd.DataFrame({"station": stations.to_numpy(), "ae": errors}).groupby(
        "station", observed=True
    )["ae"].mean()
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "macro_station_mae": float(station_mae.mean()),
        "worst_station_mae": float(station_mae.max()),
        "p95_absolute_error": float(np.quantile(errors, 0.95)),
    }


def seasonal_baseline(train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    reference = train.assign(
        month=train["datetime"].dt.month,
        hour=train["datetime"].dt.hour,
    )
    lookup = reference.groupby(["nama_pos", "month", "hour"], observed=True)[
        "tma_mdpl"
    ].median()
    fallback = reference.groupby("nama_pos", observed=True)["tma_mdpl"].median()
    key = pd.MultiIndex.from_arrays(
        [valid["nama_pos"], valid["datetime"].dt.month, valid["datetime"].dt.hour]
    )
    pred = np.array(lookup.reindex(key).to_numpy(dtype=float), dtype=float, copy=True)
    missing = np.isnan(pred)
    pred[missing] = valid.loc[missing, "nama_pos"].map(fallback).to_numpy(dtype=float)
    return pred


# --------------------------------------------------------------------------------
# Per-station harmonic baseline FEATURE (not just a comparison metric).
#
# Follow-up to the Wonogiri Dam investigation: both the CatBoost/LightGBM ensemble
# and a completely different XGBoost-residual-harmonic challenger showed the SAME
# direction of bias drift across folds, which points to a missing SIGNAL (most
# likely dam operation schedule) rather than a modeling/architecture problem.
# This can't fix that gap, but it gives the tree ensemble an explicit, smoothly
# extrapolable per-station seasonal/diurnal curve fit directly against the target
# -- something month_cat/dayofyear_sin/cos alone can't provide per-station without
# the trees having to reconstruct a station x season interaction from scratch.
#
# Leakage rule: exactly like fit_target_stats, this MUST be fit only on the
# training prefix of a given fold (or the full train set for the final fit) and
# then evaluated on fit+valid (or train+test) using ONLY calendar information.
# It never reads the target value of the row it's predicting for.
# --------------------------------------------------------------------------------

def _harmonic_design_matrix(datetime_series: pd.Series) -> np.ndarray:
    doy = datetime_series.dt.dayofyear.to_numpy(dtype=float)
    hour = datetime_series.dt.hour.to_numpy(dtype=float)
    return np.column_stack([
        np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
        np.sin(4 * np.pi * doy / 365.25), np.cos(4 * np.pi * doy / 365.25),
        np.sin(2 * np.pi * hour / 24.0), np.cos(2 * np.pi * hour / 24.0),
    ])


def fit_harmonic_baselines(fit_frame: pd.DataFrame, alpha: float) -> dict[str, Ridge]:
    """One causal Ridge harmonic curve per station, fit only on `fit_frame`."""
    baselines: dict[str, Ridge] = {}
    for station, group in fit_frame.groupby("nama_pos", observed=True):
        X = _harmonic_design_matrix(group["datetime"])
        model = Ridge(alpha=alpha)
        model.fit(X, group["tma_mdpl"].to_numpy(dtype=float))
        baselines[station] = model
    return baselines


def add_harmonic_baseline_feature(
    frame: pd.DataFrame, baselines: dict[str, Ridge], fallback_center: pd.Series
) -> pd.DataFrame:
    """Overwrites the harmonic_baseline_mdpl placeholder with causal predictions.

    Stations absent from `baselines` (shouldn't normally happen since eligibility
    filtering keeps fit/valid station sets aligned, but kept as a safety net) fall
    back to their per-station median, mirroring `seasonal_baseline`'s fallback.
    """
    out = frame.copy()
    X = _harmonic_design_matrix(out["datetime"])
    pred = np.full(len(out), np.nan, dtype=float)
    stations = out["nama_pos"].to_numpy()
    for station, model in baselines.items():
        mask = stations == station
        if mask.any():
            pred[mask] = model.predict(X[mask])
    missing = np.isnan(pred)
    if missing.any():
        pred[missing] = out.loc[missing, "nama_pos"].map(fallback_center).to_numpy(dtype=float)
    out["harmonic_baseline_mdpl"] = pred
    return out


def apply_station_features(
    fit_frame: pd.DataFrame, other_frames: Sequence[pd.DataFrame], config: ModelConfig
) -> tuple[pd.DataFrame, list[pd.DataFrame]]:
    """Single entry point for every feature that needs a causal per-station
    harmonic baseline fit on `fit_frame`: harmonic_baseline_mdpl AND the
    recency/persistence "anchor" features below. The Ridge curves are fit ONCE
    per call and reused for both, instead of fitting them twice.
    Each half is independently toggle-able and safely no-ops when disabled.
    """
    if not config.enable_harmonic_feature and not config.enable_anchor_feature:
        return fit_frame, list(other_frames)

    baselines = fit_harmonic_baselines(fit_frame, config.harmonic_ridge_alpha)
    fit_out = fit_frame
    others_out = list(other_frames)

    if config.enable_harmonic_feature:
        fallback_center = fit_frame.groupby("nama_pos", observed=True)["tma_mdpl"].median()
        fit_out = add_harmonic_baseline_feature(fit_out, baselines, fallback_center)
        others_out = [add_harmonic_baseline_feature(f, baselines, fallback_center) for f in others_out]

    if config.enable_anchor_feature:
        cutoff = fit_frame["datetime"].max()
        recent_anomaly = compute_recent_anomaly(fit_frame, baselines, config.anchor_recent_window_days)
        fit_out = add_anchor_features(fit_out, cutoff, recent_anomaly, config.anchor_tau_days)
        others_out = [add_anchor_features(f, cutoff, recent_anomaly, config.anchor_tau_days) for f in others_out]

    return fit_out, others_out


# --------------------------------------------------------------------------------
# Recency/persistence "anchor" features -- follow-up to a peer team's
# station-aware stacking result (official leaderboard 1.59656 vs our 1.60232).
#
# Their key idea, distilled: per-station anomaly relative to the seasonal norm,
# measured right at the training cutoff, times an exponential horizon-decay,
# corrects persistent per-station level bias (exactly the Wonogiri Dam pattern:
# bias magnitude/direction depends on how far into the fold you are). They
# hand-tuned a per-station alpha for this correction with OOF-based shrinkage.
#
# We take the same signal but expose it as plain FEATURES fed into the existing
# CatBoost/LightGBM ensemble instead of a hand-tuned post-hoc correction -- the
# trees decide per-station/per-region how much to trust it, using the same
# leakage discipline as harmonic_baseline_mdpl (fit only on the fold's train
# prefix, evaluated causally on fit+valid).
# --------------------------------------------------------------------------------

def compute_recent_anomaly(
    fit_frame: pd.DataFrame, baselines: dict[str, Ridge], window_days: float
) -> pd.Series:
    """Per-station mean residual (tma_mdpl - harmonic_baseline) over the last
    `window_days` of `fit_frame` -- "how far off is this station right now
    relative to its normal seasonal curve, right before we start forecasting."
    """
    X = _harmonic_design_matrix(fit_frame["datetime"])
    baseline_pred = np.full(len(fit_frame), np.nan, dtype=float)
    stations = fit_frame["nama_pos"].to_numpy()
    for station, model in baselines.items():
        mask = stations == station
        if mask.any():
            baseline_pred[mask] = model.predict(X[mask])
    residual = fit_frame["tma_mdpl"].to_numpy(dtype=float) - baseline_pred
    cutoff = fit_frame["datetime"].max()
    window_start = cutoff - pd.Timedelta(days=window_days)
    recent_mask = (fit_frame["datetime"] >= window_start).to_numpy()
    recent = pd.DataFrame({"nama_pos": stations[recent_mask], "residual": residual[recent_mask]})
    return recent.groupby("nama_pos", observed=True)["residual"].mean()


def add_anchor_features(
    frame: pd.DataFrame,
    cutoff: pd.Timestamp,
    recent_anomaly: pd.Series,
    tau_days: float,
) -> pd.DataFrame:
    out = frame.copy()
    horizon_days = (out["datetime"] - cutoff).dt.total_seconds().to_numpy() / 86_400.0
    anomaly = out["nama_pos"].map(recent_anomaly).fillna(0.0).to_numpy(dtype=float)
    out["horizon_days"] = horizon_days
    out["recent_anomaly_mdpl"] = anomaly
    out["anchor_basis_mdpl"] = np.exp(-np.abs(horizon_days) / tau_days) * anomaly
    return out


# --------------------------------------------------------------------------------
# Multi-family model builder. Every member predicts the SAME robust-normalized
# target ("z-scale"), so CatBoost and LightGBM predictions stay directly
# averageable regardless of family (senior review points 1 and 2).
# --------------------------------------------------------------------------------

def _xy_catboost(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    result = frame.loc[:, columns].copy()
    for col in set(columns) & set(CATEGORICAL_COLUMNS):
        result[col] = result[col].fillna("__MISSING__").astype(str)
    return result


def _xy_lightgbm(frame: pd.DataFrame, columns: Sequence[str], cat_columns: Sequence[str]) -> pd.DataFrame:
    """LightGBM wants pandas 'category' dtype rather than raw strings."""
    result = frame.loc[:, columns].copy()
    for col in cat_columns:
        result[col] = result[col].fillna("__MISSING__").astype(str).astype("category")
    return result


def make_catboost_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int, train_dir: Path | None = None):
    from catboost import CatBoostRegressor
    params = dict(
        iterations=iterations,
        learning_rate=member.learning_rate,
        depth=member.depth,
        loss_function=f"Huber:delta={config.huber_delta}",
        eval_metric="MAE",
        l2_leaf_reg=member.l2_leaf_reg,
        random_seed=member.seed,
        random_strength=0.8,
        bootstrap_type="Bayesian",
        bagging_temperature=0.7,
        allow_writing_files=train_dir is not None,
        thread_count=-1,
        verbose=config.verbose_every,
    )
    if train_dir is not None:
        train_dir.mkdir(parents=True, exist_ok=True)
        params.update(
            train_dir=str(train_dir),
            save_snapshot=True,
            snapshot_file="training.snapshot",
            snapshot_interval=config.snapshot_interval_seconds,
        )
    return CatBoostRegressor(**params)


def make_lightgbm_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int):
    try:
        import lightgbm as lgb
    except ImportError as exc:
        raise RuntimeError(
            "LightGBM is required for the diversified ensemble. Install requirements.txt first."
        ) from exc
    return lgb.LGBMRegressor(
        objective="huber",
        alpha=config.huber_delta,  # same Huber delta as the CatBoost members
        n_estimators=iterations,
        learning_rate=member.learning_rate,
        num_leaves=member.num_leaves,
        feature_fraction=member.feature_fraction,
        bagging_fraction=member.bagging_fraction,
        bagging_freq=1,
        min_child_samples=member.min_child_samples,
        random_state=member.seed,
        n_jobs=-1,
        verbosity=-1,
    )


def build_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int, train_dir: Path | None = None):
    if member.family == "catboost":
        return make_catboost_model(member, config, iterations, train_dir)
    if member.family == "lightgbm":
        return make_lightgbm_model(member, config, iterations)
    raise ValueError(f"Unknown model family: {member.family}")


def fit_member(
    model,
    member: "EnsembleMemberConfig",
    columns: Sequence[str],
    cat_columns: Sequence[str],
    fit_frame: pd.DataFrame,
    y_fit: np.ndarray,
    weights: np.ndarray,
    valid_frame: pd.DataFrame | None = None,
    y_valid: np.ndarray | None = None,
    early_stopping_rounds: int | None = None,
):
    """Fits one ensemble member. Returns (fitted_model, best_iteration_used)."""
    if member.family == "catboost":
        X_fit = _xy_catboost(fit_frame, columns)
        eval_set = None
        if valid_frame is not None:
            eval_set = (_xy_catboost(valid_frame, columns), y_valid)
        model.fit(
            X_fit, y_fit,
            cat_features=list(cat_columns),
            sample_weight=weights,
            eval_set=eval_set,
            early_stopping_rounds=early_stopping_rounds,
            use_best_model=eval_set is not None,
        )
        if eval_set is not None:
            return model, int(model.get_best_iteration()) + 1
        return model, int(model.tree_count_)

    import lightgbm as lgb
    X_fit = _xy_lightgbm(fit_frame, columns, cat_columns)
    callbacks = [lgb.log_evaluation(period=0)]
    eval_set = None
    if valid_frame is not None:
        X_valid = _xy_lightgbm(valid_frame, columns, cat_columns)
        eval_set = [(X_valid, y_valid)]
        callbacks.append(lgb.early_stopping(early_stopping_rounds, verbose=False))
    model.fit(
        X_fit, y_fit,
        sample_weight=weights,
        eval_set=eval_set,
        categorical_feature=list(cat_columns),
        callbacks=callbacks,
    )
    best_iteration = getattr(model, "best_iteration_", None)
    return model, int(best_iteration) if best_iteration else int(model.n_estimators)


def predict_member(model, member: "EnsembleMemberConfig", frame: pd.DataFrame, columns: Sequence[str], cat_columns: Sequence[str]) -> np.ndarray:
    if member.family == "catboost":
        return model.predict(_xy_catboost(frame, columns))
    return model.predict(_xy_lightgbm(frame, columns, cat_columns))


def member_feature_importance(model, member: "EnsembleMemberConfig", columns: Sequence[str]) -> dict:
    if member.family == "catboost":
        values = model.get_feature_importance()
    else:
        raw = np.asarray(model.feature_importances_, dtype=float)
        values = raw / raw.sum() * 100.0 if raw.sum() > 0 else raw
    return dict(zip(columns, (float(v) for v in values)))


## 7. Run tracking dan handoff untuk penyempurnaan model

Setiap training membuat direktori dan ZIP bertimestamp UTC. Arsip `in_progress` diperbarui selama training; arsip `complete` atau `failed` menjadi handoff akhir untuk analisis Codex.


In [ ]:
HANDOFF_SCHEMA_VERSION = "1.0"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def create_run_directory(output_root: Path, mode: str) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = output_root / f"handoff_{stamp}_{mode}"
    suffix = 1
    while run_dir.exists():
        run_dir = output_root / f"handoff_{stamp}_{mode}_{suffix:02d}"
        suffix += 1
    for child in ("data", "metrics", "models", "training_progress"):
        (run_dir / child).mkdir(parents=True, exist_ok=True)
    return run_dir


def update_manifest(run_dir: Path, **changes) -> dict:
    path = run_dir / "manifest.json"
    manifest = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    manifest.update(changes)
    manifest["updated_at_utc"] = utc_now_iso()
    write_json(path, manifest)
    return manifest


def attach_run_logger(run_dir: Path) -> None:
    for handler in list(LOGGER.handlers):
        if getattr(handler, "_tma_handoff_handler", False):
            LOGGER.removeHandler(handler)
            handler.close()
    handler = logging.FileHandler(run_dir / "run.log", encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    handler._tma_handoff_handler = True
    LOGGER.addHandler(handler)
    LOGGER.setLevel(logging.INFO)


def runtime_versions() -> dict:
    packages = [
        "catboost", "lightgbm", "joblib", "geopandas", "numpy", "pandas",
        "pyarrow", "scikit-learn", "shapely",
    ]
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }


def input_inventory(paths: Paths) -> list[dict]:
    river_sidecars = sorted(paths.rivers.parent.glob(f"{paths.rivers.stem}.*"))
    files = [paths.train, paths.test, paths.sample_submission, paths.coordinates, paths.environment, *river_sidecars]
    return [
        {"path": str(path), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None}
        for path in files
    ]


def save_frame(frame: pd.DataFrame, path_without_suffix: Path) -> Path:
    """Prefer compressed parquet; fall back to gzip CSV if pyarrow is absent."""
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = path_without_suffix.with_suffix(".parquet")
    try:
        frame.to_parquet(parquet_path, index=False, compression="zstd")
        return parquet_path
    except (ImportError, ModuleNotFoundError):
        csv_path = path_without_suffix.with_suffix(".csv.gz")
        frame.to_csv(csv_path, index=False, compression="gzip")
        return csv_path


def environment_drift_report(
    train_features: pd.DataFrame, test_features: pd.DataFrame, columns: list[str]
) -> pd.DataFrame:
    numeric = [c for c in columns if pd.api.types.is_numeric_dtype(train_features[c])]
    rows = []
    for col in numeric:
        train_values = train_features[col].replace([np.inf, -np.inf], np.nan)
        test_values = test_features[col].replace([np.inf, -np.inf], np.nan)
        train_std = float(train_values.std())
        rows.append({
            "feature": col,
            "train_missing_pct": float(train_values.isna().mean() * 100),
            "test_missing_pct": float(test_values.isna().mean() * 100),
            "train_mean": float(train_values.mean()),
            "test_mean": float(test_values.mean()),
            "train_std": train_std,
            "test_std": float(test_values.std()),
            "standardized_mean_shift": float((test_values.mean() - train_values.mean()) / train_std) if train_std > 0 else np.nan,
            "train_q01": float(train_values.quantile(0.01)),
            "train_q50": float(train_values.quantile(0.50)),
            "train_q99": float(train_values.quantile(0.99)),
            "test_q01": float(test_values.quantile(0.01)),
            "test_q50": float(test_values.quantile(0.50)),
            "test_q99": float(test_values.quantile(0.99)),
        })
    return pd.DataFrame(rows)


def write_handoff_readme(run_dir: Path) -> None:
    text = """# TMA model handoff\n\nThis archive is a reproducible diagnostic handoff for the next model iteration.\n\nCore files:\n- manifest.json: run identity, timestamps, status, and stage.\n- config.json / runtime_versions.json / input_inventory.json: reproducibility context (includes the exact ensemble member configs).\n- data/train_features.* and test_features.*: exact model matrices when enabled. NOTE: if enable_harmonic_feature=True, the harmonic_baseline_mdpl column here is a NaN PLACEHOLDER -- real values are fold-specific and computed on the fly inside run_validation/fit_final_and_submit.\n- data/train_sample_diagnostics.*: normalized target, sample weights, and spike flags.\n- metrics/oof_predictions.*: leakage-safe ENSEMBLE validation predictions and residuals.\n- metrics/validation_metrics.csv: global fold scores (ensemble) and baseline comparison.\n- metrics/member_metrics.csv: per-member (CatBoost/LightGBM) fold scores, for diversity diagnostics.\n- metrics/fold_station_metrics.csv: failure analysis by station.\n- metrics/residual_slices.csv: residuals by station, month, and hour.\n- metrics/feature_importance.csv: importance by fold/member; feed this back as feature_importance_source to enable pruning.\n- metrics/environment_drift.csv: train-to-test feature distribution shift.\n- metrics/final_feature_importance.csv: importance from the final fit, per member.\n- recommended_iterations.json: median best-iteration PER ensemble member (not a single number).\n- training_progress/: live CatBoost logs and snapshots for every fold/member.\n- models/: fold and final CatBoost (.cbm) and LightGBM (.joblib) models.\n- submission.csv: exact competition output when final fit ran.\n- submission_quantiles.csv: optional CatBoost MultiQuantile prediction intervals, when enable_quantile_model=True.\n- run.log / error_traceback.txt: orchestration log and failure context.\n\nSeparately, MODE=\"ablation\" and MODE=\"tune\" write lightweight, non-zipped diagnostic\nfolders (ablation_<timestamp>/, tune_<timestamp>/) next to the handoff runs -- use\nthose to decide use_spike_weights/prune_features/hyperparameters BEFORE committing to\na full validate/fit run.\n\nUse OOF predictions—not training error—as the primary basis for refinements.\n"""
    (run_dir / "HANDOFF_README.md").write_text(text, encoding="utf-8")


def create_handoff_zip(run_dir: Path, status: str) -> Path:
    if status not in {"in_progress", "complete", "failed"}:
        raise ValueError("Invalid handoff ZIP status")
    destination = run_dir.parent / f"{run_dir.name}_{status}.zip"
    temporary = destination.with_suffix(".zip.tmp")
    if temporary.exists():
        temporary.unlink()
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(run_dir.rglob("*")):
            if path.is_file():
                already_compressed = path.suffix.lower() in {".parquet", ".gz", ".cbm", ".snapshot"}
                archive.write(
                    path, arcname=str(Path(run_dir.name) / path.relative_to(run_dir)),
                    compress_type=zipfile.ZIP_STORED if already_compressed else zipfile.ZIP_DEFLATED,
                )
    temporary.replace(destination)
    if status in {"complete", "failed"}:
        in_progress = run_dir.parent / f"{run_dir.name}_in_progress.zip"
        if in_progress.exists():
            in_progress.unlink()
    return destination


def save_pretraining_handoff(
    run_dir: Path,
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    upstream_weights: pd.DataFrame,
    config: ModelConfig,
    save_feature_matrices: bool,
) -> None:
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    write_json(run_dir / "feature_columns.json", {"features": columns, "categorical": cat_columns})
    upstream_weights.to_csv(run_dir / "data" / "upstream_station_weights.csv")
    stats = fit_target_stats(train_features)
    stats.to_csv(run_dir / "data" / "target_normalization.csv")
    sample_weights, _ = compute_sample_weights(train_features, config.use_spike_weights)
    diagnostics = train_features[KEYS + ["tma_mdpl"]].copy()
    diagnostics["target_normalized"] = normalize_target(train_features, stats)
    diagnostics["sample_weight"] = sample_weights
    diagnostics["isolated_spike_flag"] = (sample_weights < 1).astype("int8")
    save_frame(diagnostics, run_dir / "data" / "train_sample_diagnostics")
    drift = environment_drift_report(train_features, test_features, columns)
    drift.to_csv(run_dir / "metrics" / "environment_drift.csv", index=False)
    if save_feature_matrices:
        train_export_columns = list(dict.fromkeys(KEYS + ["tma_mdpl"] + columns))
        test_export_columns = list(dict.fromkeys(["id"] + KEYS + columns))
        train_export = train_features[train_export_columns].copy()
        test_export = test_features[test_export_columns].copy()
        save_frame(train_export, run_dir / "data" / "train_features")
        save_frame(test_export, run_dir / "data" / "test_features")
    update_manifest(run_dir, stage="features_ready", feature_count=len(columns), status="in_progress")
    create_handoff_zip(run_dir, "in_progress")


## 8. Rolling-origin validation: skor ensemble penuh per fold

Empat fold memakai horizon panjang agar menyerupai deployment 242 hari. Setiap fold melatih dan merata-ratakan **seluruh anggota ensemble** (bukan satu seed) sebelum menghitung metrik, jadi angka yang dilaporkan cocok dengan submission final. Cell ini hanya mendefinisikan fungsi; belum menjalankan training.


In [ ]:
def run_validation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds: Sequence[tuple] = FOLDS,
    members: Sequence[EnsembleMemberConfig] | None = None,
    write_artifacts: bool = True,
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Rolling-origin validation that scores the FULL ensemble per fold.

    Senior review point 1: this used to train only ``config.seeds[0]`` per fold, so
    the reported RMSE didn't match the 3-seed ensemble actually submitted. Now every
    member is trained and averaged on the z-scale before computing metrics, exactly
    like ``fit_final_and_submit`` does.

    ``folds``/``members`` subsets and ``write_artifacts=False`` let the ablation and
    hyperparameter-sweep cells reuse this exact function cheaply instead of
    duplicating the fold/leakage logic.
    """
    members = tuple(members) if members is not None else config.ensemble
    records: list[dict] = []
    member_records: list[dict] = []
    best_iterations: dict[str, list[int]] = {m.name: [] for m in members}
    oof_parts: list[pd.DataFrame] = []
    station_records: list[dict] = []
    importance_records: list[dict] = []
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]

    for fold_name, train_end, valid_start, valid_end in folds:
        fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
        valid = train_features[
            train_features["datetime"].between(
                pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
            )
        ].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit = fit[fit["nama_pos"].isin(eligible)]
        valid = valid[valid["nama_pos"].isin(eligible)]
        if fit.empty or valid.empty:
            LOGGER.warning("Skipping empty fold %s", fold_name)
            continue

        stats = fit_target_stats(fit)
        y_fit = normalize_target(fit, stats)
        y_valid_z = normalize_target(valid, stats)
        sample_weights, spike_count = compute_sample_weights(fit, config.use_spike_weights)
        fit, (valid,) = apply_station_features(fit, (valid,), config)

        member_preds_z = []
        member_pred_columns: dict[str, np.ndarray] = {}
        for member in members:
            progress_dir = (
                artifacts / "training_progress" / f"fold_{fold_name}_{member.name}"
                if write_artifacts else None
            )
            model = build_model(member, config, iterations=config.default_iterations, train_dir=progress_dir)
            model, best_iteration = fit_member(
                model, member, columns, cat_columns,
                fit, y_fit, sample_weights,
                valid_frame=valid, y_valid=y_valid_z,
                early_stopping_rounds=config.early_stopping_rounds,
            )
            best_iterations[member.name].append(best_iteration)
            pred_z_member = predict_member(model, member, valid, columns, cat_columns)
            member_preds_z.append(pred_z_member)

            member_pred = denormalize_prediction(pred_z_member, valid["nama_pos"], stats)
            member_pred_columns[member.name] = member_pred
            member_metrics = metrics_frame(valid["tma_mdpl"].to_numpy(), member_pred, valid["nama_pos"])
            member_records.append({
                "fold": fold_name, "member": member.name, "family": member.family,
                "best_iteration": best_iteration,
                **{f"model_{k}": v for k, v in member_metrics.items()},
            })

            if write_artifacts:
                for feature, importance in member_feature_importance(model, member, columns).items():
                    importance_records.append({
                        "fold": fold_name, "member": member.name, "feature": feature, "importance": importance,
                    })
                if member.family == "catboost":
                    model.save_model(str(artifacts / "models" / f"validation_{fold_name}_{member.name}.cbm"))

        pred_z = np.mean(member_preds_z, axis=0)
        pred = denormalize_prediction(pred_z, valid["nama_pos"], stats)
        model_metrics = metrics_frame(valid["tma_mdpl"].to_numpy(), pred, valid["nama_pos"])
        baseline_pred = seasonal_baseline(fit, valid)
        baseline_metrics = metrics_frame(
            valid["tma_mdpl"].to_numpy(), baseline_pred, valid["nama_pos"]
        )

        if write_artifacts:
            fold_stats = stats.reset_index()
            fold_stats.to_csv(artifacts / "data" / f"target_stats_{fold_name}.csv", index=False)

        oof = valid[KEYS + ["tma_mdpl"]].copy()
        oof["fold"] = fold_name
        oof["target_normalized"] = y_valid_z
        oof["prediction_normalized"] = pred_z
        oof["prediction"] = pred
        oof["baseline_prediction"] = baseline_pred
        for member_name, member_prediction in member_pred_columns.items():
            oof[f"prediction_{member_name}"] = member_prediction
        oof["error"] = oof["prediction"] - oof["tma_mdpl"]
        oof["absolute_error"] = oof["error"].abs()
        oof_parts.append(oof)

        for station, station_frame in oof.groupby("nama_pos", observed=True):
            station_records.append({
                "fold": fold_name,
                "nama_pos": station,
                "rows": len(station_frame),
                "mae": float(station_frame["absolute_error"].mean()),
                "rmse": float(np.sqrt(np.mean(station_frame["error"] ** 2))),
                "bias": float(station_frame["error"].mean()),
            })

        records.append(
            {
                "fold": fold_name,
                "train_rows": len(fit),
                "valid_rows": len(valid),
                "valid_stations": valid["nama_pos"].nunique(),
                "isolated_spikes_downweighted": spike_count,
                "use_spike_weights": config.use_spike_weights,
                "ensemble_members": ",".join(m.name for m in members),
                **{f"model_{k}": v for k, v in model_metrics.items()},
                **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
            }
        )
        LOGGER.info("Completed fold %s (ensemble of %d): %s", fold_name, len(members), model_metrics)

        if write_artifacts:
            cumulative_oof = pd.concat(oof_parts, ignore_index=True)
            save_frame(cumulative_oof, artifacts / "metrics" / "oof_predictions")
            pd.DataFrame(records).to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
            pd.DataFrame(member_records).to_csv(artifacts / "metrics" / "member_metrics.csv", index=False)
            pd.DataFrame(station_records).to_csv(artifacts / "metrics" / "fold_station_metrics.csv", index=False)
            pd.DataFrame(importance_records).to_csv(artifacts / "metrics" / "feature_importance.csv", index=False)
            update_manifest(artifacts, stage=f"validation_{fold_name}_complete", completed_folds=[r["fold"] for r in records])
            create_handoff_zip(artifacts, "in_progress")

    if not records:
        raise RuntimeError("No validation fold was runnable")
    report = pd.DataFrame(records)
    recommended = {name: int(np.median(iters)) for name, iters in best_iterations.items() if iters}

    if write_artifacts:
        artifacts.mkdir(parents=True, exist_ok=True)
        report.to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
        pd.DataFrame(member_records).to_csv(artifacts / "metrics" / "member_metrics.csv", index=False)
        oof_all = pd.concat(oof_parts, ignore_index=True)
        residual_slices = (
            oof_all.assign(
                month=oof_all["datetime"].dt.month,
                hour=oof_all["datetime"].dt.hour,
            )
            .groupby(["nama_pos", "month", "hour"], observed=True)
            .agg(
                rows=("error", "size"),
                mae=("absolute_error", "mean"),
                bias=("error", "mean"),
                p95_absolute_error=("absolute_error", lambda s: s.quantile(0.95)),
            )
            .reset_index()
        )
        residual_slices.to_csv(artifacts / "metrics" / "residual_slices.csv", index=False)
        write_json(artifacts / "recommended_iterations.json", recommended)
        update_manifest(artifacts, stage="validation_complete", recommended_iterations=recommended)
        create_handoff_zip(artifacts, "in_progress")

    return report, recommended


## 8b. Ablasi: spike-downweighting dan feature pruning on/off

`run_ablation_study` menjawab kritik "spike-downweighting dan feature pruning belum divalidasi secara empiris". Fungsi ini memakai satu model cepat (LightGBM) pada 2 fold terakhir untuk membandingkan kombinasi `use_spike_weights` on/off dan fitur penuh vs. hasil pruning dari `feature_importance.csv`. Jauh lebih murah daripada full ensemble x 4 fold, cukup untuk memutuskan toggle mana yang dipakai di run sesungguhnya.


In [ ]:
def run_ablation_study(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    ablation_folds: Sequence[tuple] = FOLDS[-2:],
    ablation_member: EnsembleMemberConfig | None = None,
    feature_importance_source: str | Path | None = None,
) -> pd.DataFrame:
    """Cheap on/off ablation for spike-downweighting and feature pruning.

    Senior review point 3: both mechanisms were shipped on intuition, without a
    measured effect on validation metrics. This runs a SINGLE fast member (a
    LightGBM member by default) on only the two most recent folds -- the ones
    closest in length/position to the real test horizon -- which is cheap enough to
    run before committing to a full 5-member x 4-fold validation.

    Reads ``feature_importance_source`` (typically a prior run's
    ``metrics/feature_importance.csv``) to build the pruned variants; if it isn't
    available yet, only the spike-weight ablation runs (feature pruning needs a
    first full validation pass to exist).
    """
    member = ablation_member or next(m for m in config.ensemble if m.family == "lightgbm")
    pruned_columns, prune_report = apply_feature_pruning(
        columns, feature_importance_source, config.prune_keep_fraction
    )

    variants = [
        {"name": "baseline_full_features_spike_on", "use_spike_weights": True, "columns": columns},
        {"name": "no_spike_weight_full_features", "use_spike_weights": False, "columns": columns},
    ]
    if prune_report["applied"]:
        variants += [
            {"name": "pruned_features_spike_on", "use_spike_weights": True, "columns": pruned_columns},
            {"name": "pruned_features_no_spike", "use_spike_weights": False, "columns": pruned_columns},
        ]
    else:
        LOGGER.warning(
            "No usable feature_importance_source; skipping pruning variants. Run "
            "MODE='validate' once, then point FEATURE_IMPORTANCE_SOURCE at the "
            "resulting metrics/feature_importance.csv to unlock pruning ablation."
        )

    rows = []
    for variant in variants:
        variant_config = ModelConfig(
            ensemble=config.ensemble,
            default_iterations=config.default_iterations,
            huber_delta=config.huber_delta,
            early_stopping_rounds=config.early_stopping_rounds,
            verbose_every=config.verbose_every,
            upstream_decay_km=config.upstream_decay_km,
            use_spike_weights=variant["use_spike_weights"],
        )
        report, _ = run_validation(
            train_features, variant["columns"], variant_config, artifacts,
            folds=ablation_folds, members=(member,), write_artifacts=False,
        )
        keep_cols = [c for c in report.columns if c.startswith("model_") or c == "fold"]
        summary = report[keep_cols].copy()
        summary["variant"] = variant["name"]
        summary["n_features"] = len(variant["columns"])
        rows.append(summary)

    result = pd.concat(rows, ignore_index=True)
    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(artifacts / "ablation_results.csv", index=False)
    write_json(artifacts / "feature_pruning_report.json", prune_report)

    agg = result.groupby("variant", observed=True)[
        ["model_mae", "model_rmse", "model_macro_station_mae", "model_worst_station_mae"]
    ].mean()
    LOGGER.info(
        "Ablation summary using member=%s over %d fold(s):\n%s",
        member.name, len(ablation_folds), agg.sort_values("model_rmse").to_string(),
    )
    return result


## 8c. Hyperparameter sweep ringan (CatBoost)

`run_hyperparam_sweep` menjawab kritik "depth=8, learning_rate=0.035, l2_leaf_reg=10.0 tidak punya jejak tuning". Bukan pengganti search penuh — cuma menguji beberapa kombinasi depth/learning_rate/l2 pada fold `jan_2025` (paling mirip panjang/posisi kalender dengan horizon test asli) supaya keputusan hyperparameter berbasis angka, bukan tebakan.


In [ ]:
def run_hyperparam_sweep(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    sweep_fold: tuple = FOLDS[-1],
    grid: Sequence[dict] | None = None,
) -> pd.DataFrame:
    """Lightweight, single-fold CatBoost hyperparameter sanity check.

    Senior review point 4: `depth=8, learning_rate=0.035, l2_leaf_reg=10.0` had no
    documented search behind them. This is not a substitute for a proper search --
    it tests a handful of depth/learning_rate/l2 combinations on the fold most
    representative of the real test horizon (``jan_2025`` by default: same length
    and calendar position as the actual 19 Sep 2025 - 18 Mei 2026 test window) so a
    hyperparameter decision is backed by a number instead of a guess.
    """
    grid = grid or [
        {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
        {"depth": 8, "learning_rate": 0.035, "l2_leaf_reg": 10.0},
        {"depth": 8, "learning_rate": 0.02, "l2_leaf_reg": 10.0},
        {"depth": 10, "learning_rate": 0.025, "l2_leaf_reg": 14.0},
    ]
    fold_name, train_end, valid_start, valid_end = sweep_fold
    fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
    valid = train_features[
        train_features["datetime"].between(
            pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
        )
    ].copy()
    eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
    fit = fit[fit["nama_pos"].isin(eligible)]
    valid = valid[valid["nama_pos"].isin(eligible)]
    if fit.empty or valid.empty:
        raise RuntimeError(f"Sweep fold {fold_name} has no data; check FOLDS/sweep_fold")

    stats = fit_target_stats(fit)
    y_fit = normalize_target(fit, stats)
    y_valid_z = normalize_target(valid, stats)
    weights, _ = compute_sample_weights(fit, config.use_spike_weights)
    fit, (valid,) = apply_station_features(fit, (valid,), config)
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]

    rows = []
    for params in grid:
        member = EnsembleMemberConfig(name="sweep", family="catboost", seed=17, **params)
        model = build_model(member, config, iterations=config.default_iterations, train_dir=None)
        model, best_iteration = fit_member(
            model, member, columns, cat_columns, fit, y_fit, weights,
            valid_frame=valid, y_valid=y_valid_z,
            early_stopping_rounds=config.early_stopping_rounds,
        )
        pred = denormalize_prediction(
            predict_member(model, member, valid, columns, cat_columns), valid["nama_pos"], stats
        )
        metrics = metrics_frame(valid["tma_mdpl"].to_numpy(), pred, valid["nama_pos"])
        rows.append({"fold": fold_name, **params, "best_iteration": best_iteration, **metrics})
        LOGGER.info("Sweep %s: %s", params, metrics)

    result = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
    return result


## 9. Final ensemble fit, submission, dan model quantile opsional

Setiap anggota ensemble dilatih dengan jumlah iterasi rekomendasinya sendiri (median hasil validasi per member, bukan satu angka global), lalu prediksi dirata-ratakan pada skala target ternormalisasi. Bila `enable_quantile_model=True`, satu CatBoost `MultiQuantile` tambahan dilatih untuk interval prediksi (`submission_quantiles.csv`) di luar titik prediksi utama. Cell ini hanya mendefinisikan fungsi.


In [ ]:
def fit_quantile_model(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    cat_columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    weights: np.ndarray,
    y: np.ndarray,
) -> Path:
    """Optional CatBoost MultiQuantile model.

    Senior review point 4: point predictions alone don't communicate risk for a
    flood-relevant forecast. This trains one extra CatBoost model with
    ``MultiQuantile`` loss on the SAME robust-normalized target as the main
    ensemble, and writes a prediction-interval CSV alongside ``submission.csv``.
    Off by default (``config.enable_quantile_model``); it roughly doubles the final
    training time of one CatBoost member.
    """
    from catboost import CatBoostRegressor
    alpha = ",".join(str(q) for q in config.quantile_levels)
    model = CatBoostRegressor(
        iterations=config.default_iterations,
        learning_rate=0.03,
        depth=8,
        loss_function=f"MultiQuantile:alpha={alpha}",
        random_seed=97,
        l2_leaf_reg=10.0,
        thread_count=-1,
        verbose=config.verbose_every,
    )
    model.fit(_xy_catboost(train_features, columns), y, cat_features=cat_columns, sample_weight=weights)
    pred_z = np.asarray(model.predict(_xy_catboost(test_features, columns)))
    if pred_z.ndim == 1:
        pred_z = pred_z.reshape(-1, len(config.quantile_levels))

    stats = fit_target_stats(train_features)
    center = test_features["nama_pos"].map(stats["center"]).to_numpy(dtype=float)
    scale = test_features["nama_pos"].map(stats["scale"]).to_numpy(dtype=float)

    quantile_frame = test_features[["id"]].copy()
    for i, q in enumerate(config.quantile_levels):
        quantile_frame[f"tma_mdpl_q{int(round(q * 100)):02d}"] = center + scale * pred_z[:, i]

    output_path = artifacts / "submission_quantiles.csv"
    quantile_frame.to_csv(output_path, index=False)
    model.save_model(str(artifacts / "models" / "catboost_quantile.cbm"))
    return output_path


def fit_final_and_submit(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    sample: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    iterations_by_member: dict[str, int],
) -> Path:
    artifacts.mkdir(parents=True, exist_ok=True)
    stats = fit_target_stats(train_features)
    y = normalize_target(train_features, stats)
    weights, spike_count = compute_sample_weights(train_features, config.use_spike_weights)
    train_features, (test_features,) = apply_station_features(train_features, (test_features,), config)
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    predictions = []
    final_importance_records = []

    for member in config.ensemble:
        iterations = iterations_by_member.get(member.name, config.default_iterations)
        progress_dir = artifacts / "training_progress" / f"final_{member.name}"
        model = build_model(
            member, config, iterations=iterations,
            train_dir=progress_dir if member.family == "catboost" else None,
        )
        model, _ = fit_member(model, member, columns, cat_columns, train_features, y, weights)

        if member.family == "catboost":
            model.save_model(str(artifacts / "models" / f"catboost_{member.name}.cbm"))
        else:
            import joblib
            joblib.dump(model, artifacts / "models" / f"lightgbm_{member.name}.joblib")

        predictions.append(predict_member(model, member, test_features, columns, cat_columns))
        for feature, importance in member_feature_importance(model, member, columns).items():
            final_importance_records.append({
                "member": member.name, "family": member.family, "feature": feature, "importance": importance,
            })
        pd.DataFrame(final_importance_records).to_csv(
            artifacts / "metrics" / "final_feature_importance.csv", index=False
        )
        completed = [
            m.name for m in config.ensemble
            if (artifacts / "models" / f"catboost_{m.name}.cbm").exists()
            or (artifacts / "models" / f"lightgbm_{m.name}.joblib").exists()
        ]
        update_manifest(artifacts, stage=f"final_{member.name}_complete", completed_final_members=completed)
        create_handoff_zip(artifacts, "in_progress")

    member_weight_vector = np.array([V3_MEMBER_WEIGHTS.get(m.name, 0.0) for m in config.ensemble], dtype=float)
    if member_weight_vector.sum() <= 0:
        member_weight_vector[:] = 1.0 / len(member_weight_vector)
    else:
        member_weight_vector /= member_weight_vector.sum()
    mean_z = np.average(np.vstack(predictions), axis=0, weights=member_weight_vector)
    prediction = denormalize_prediction(mean_z, test_features["nama_pos"], stats)

    quantile_path = None
    if config.enable_quantile_model:
        quantile_path = fit_quantile_model(
            train_features, test_features, columns, cat_columns, config, artifacts, weights, y
        )

    ordered = pd.DataFrame(
        {
            "_row_order": test_features["_row_order"],
            "id": test_features["id"],
            "tma_mdpl": prediction,
        }
    ).sort_values("_row_order")
    submission = sample[["id"]].merge(
        ordered[["id", "tma_mdpl"]], on="id", how="left", validate="one_to_one"
    )
    if submission["tma_mdpl"].isna().any() or not np.isfinite(submission["tma_mdpl"]).all():
        raise ValueError("Submission contains missing/non-finite predictions")
    output_path = artifacts / "submission.csv"
    submission.to_csv(output_path, index=False)

    stats.to_csv(artifacts / "data" / "target_normalization.csv")
    (artifacts / "feature_columns.json").write_text(
        json.dumps({"features": columns, "categorical": cat_columns}, indent=2),
        encoding="utf-8",
    )
    metadata = {
        "iterations_by_member": iterations_by_member,
        "ensemble": [asdict(m) for m in config.ensemble],
        "use_spike_weights": config.use_spike_weights,
        "isolated_spikes_downweighted": spike_count,
        "train_rows": len(train_features),
        "test_rows": len(test_features),
        "quantile_output": str(quantile_path) if quantile_path else None,
    }
    (artifacts / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    update_manifest(artifacts, stage="final_fit_complete", final_iterations_by_member=iterations_by_member)
    create_handoff_zip(artifacts, "in_progress")
    return output_path


## 10. Orkestrator notebook dan mode eksekusi

`run_pipeline` sekarang punya 6 mode: `audit`, `tune`, `validate`, `ablation`, `fit`, `all`. Mode `tune` dan `ablation` menulis folder diagnostik ringan (`tune_<timestamp>/`, `ablation_<timestamp>/`) tanpa ZIP handoff besar — dipakai untuk memutuskan konfigurasi sebelum commit ke `validate`/`fit`/`all` yang penuh.


In [ ]:
def find_latest_recommended_iterations(output_dir: Path) -> dict[str, int] | None:
    candidates = sorted(
        output_dir.glob("handoff_*/recommended_iterations.json"),
        key=lambda path: path.stat().st_mtime, reverse=True,
    )
    if not candidates:
        return None
    return json.loads(candidates[0].read_text(encoding="utf-8"))


def run_pipeline(
    mode: str = "audit",
    data_dir: Path = Path("datasets"),
    output_dir: Path = Path("artifacts"),
    config: ModelConfig | None = None,
    iterations_by_member: dict[str, int] | None = None,
    save_feature_matrices: bool = True,
) -> dict:
    """Run a Kaggle/local mode and create a timestamped diagnostic handoff.

    Modes:
      audit     - input validation only, no training.
      ablation  - cheap on/off ablation of spike-weighting / feature-pruning.
                  Writes ablation_<timestamp>/ablation_results.csv, no ZIP.
      tune      - light single-fold CatBoost hyperparameter sweep.
                  Writes tune_<timestamp>/hyperparam_sweep.csv, no ZIP.
      validate  - full ensemble rolling-origin validation, no final fit.
      fit       - final ensemble fit + submission using the iterations passed in,
                  or the latest previously recommended per-member iterations.
      all       - validate, then fit, then a complete handoff ZIP.

    Run order recommendation: audit -> tune -> validate -> ablation
    (now that feature_importance.csv exists) -> fit/all with the chosen config.
    """
    if mode not in {"audit", "ablation", "tune", "validate", "fit", "all"}:
        raise ValueError("mode must be one of: audit, ablation, tune, validate, fit, all")
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )
    paths = Paths(root=Path(data_dir), artifacts=Path(output_dir))
    config = config or ModelConfig()
    train, test, sample, coordinates, environment = load_inputs(paths)
    audit = audit_inputs(train, test, sample, coordinates, environment)
    LOGGER.info("Input audit passed:\n%s", json.dumps(audit, indent=2))
    result = {"mode": mode, "audit": audit}
    if mode == "audit":
        return result

    if mode in {"ablation", "tune"}:
        # Cheap diagnostic modes: build features once, run a lightweight study, and
        # write results next to (not inside) the heavy handoff run directories --
        # no point zipping up a multi-GB handoff for a sanity-check run.
        train_features, test_features, upstream_weights = build_feature_tables(
            train, test, coordinates, environment, paths, config
        )
        columns = feature_columns(train_features)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        light_dir = paths.artifacts / f"{mode}_{stamp}"
        light_dir.mkdir(parents=True, exist_ok=True)
        if mode == "ablation":
            table = run_ablation_study(
                train_features, columns, config, light_dir,
                feature_importance_source=config.feature_importance_source,
            )
            result["ablation_results"] = table
            result["ablation_dir"] = str(light_dir)
        else:
            table = run_hyperparam_sweep(train_features, columns, config)
            table.to_csv(light_dir / "hyperparam_sweep.csv", index=False)
            result["sweep_results"] = table
            result["sweep_dir"] = str(light_dir)
        return result

    run_dir = create_run_directory(paths.artifacts, mode)
    attach_run_logger(run_dir)
    result["run_id"] = run_dir.name
    result["run_dir"] = str(run_dir)
    manifest = {
        "schema_version": HANDOFF_SCHEMA_VERSION,
        "run_id": run_dir.name,
        "mode": mode,
        "status": "in_progress",
        "stage": "initialized",
        "started_at_utc": utc_now_iso(),
        "input_dir": str(paths.root),
        "output_root": str(paths.artifacts),
        "ensemble": [m.name for m in config.ensemble],
        "use_spike_weights": config.use_spike_weights,
        "prune_features": config.prune_features,
        "core_handoff_files": [
            "manifest.json", "config.json", "runtime_versions.json",
            "input_inventory.json", "data/train_features.*",
            "data/test_features.*", "data/train_sample_diagnostics.*",
            "metrics/environment_drift.csv", "run.log",
        ],
        "mode_dependent_files": [
            "metrics/oof_predictions.*",
            "metrics/validation_metrics.csv", "metrics/member_metrics.csv",
            "metrics/fold_station_metrics.csv", "metrics/residual_slices.csv",
            "metrics/feature_importance.csv", "metrics/final_feature_importance.csv",
            "training_progress/", "models/", "submission.csv", "submission_quantiles.csv",
        ],
    }
    write_json(run_dir / "manifest.json", manifest)
    write_json(run_dir / "data_audit.json", audit)
    write_json(run_dir / "runtime_versions.json", runtime_versions())
    write_json(run_dir / "input_inventory.json", {"files": input_inventory(paths)})
    write_json(run_dir / "config.json", {
        "ensemble": [asdict(m) for m in config.ensemble],
        "folds": FOLDS,
        "mode": mode,
        "huber_delta": config.huber_delta,
        "early_stopping_rounds": config.early_stopping_rounds,
        "use_spike_weights": config.use_spike_weights,
        "prune_features": config.prune_features,
        "feature_importance_source": config.feature_importance_source,
        "prune_keep_fraction": config.prune_keep_fraction,
        "enable_quantile_model": config.enable_quantile_model,
        "quantile_levels": config.quantile_levels,
        "requested_iterations_by_member": iterations_by_member,
        "save_feature_matrices": save_feature_matrices,
    })
    write_handoff_readme(run_dir)

    try:
        train_features, test_features, upstream_weights = build_feature_tables(
            train, test, coordinates, environment, paths, config
        )
        columns = feature_columns(train_features)
        if config.prune_features:
            columns, prune_report = apply_feature_pruning(
                columns, config.feature_importance_source, config.prune_keep_fraction
            )
            write_json(run_dir / "feature_pruning_report.json", prune_report)
        save_pretraining_handoff(
            run_dir, train_features, test_features, columns, upstream_weights, config,
            save_feature_matrices=save_feature_matrices,
        )
        LOGGER.info("Built %d leakage-safe features", len(columns))
        result["feature_count"] = len(columns)
        recommended = None
        if mode in {"validate", "all"}:
            update_manifest(
                run_dir, stage="validation_training_started",
                training_started_at_utc=utc_now_iso(),
            )
            report, recommended = run_validation(
                train_features, columns, config, run_dir
            )
            result["validation"] = report
            result["recommended_iterations"] = recommended
            LOGGER.info("Validation report:\n%s", report.to_string(index=False))
        if mode in {"fit", "all"}:
            prior_recommended = find_latest_recommended_iterations(paths.artifacts)
            if iterations_by_member is not None:
                final_iterations, iteration_source = iterations_by_member, "explicit"
            elif recommended is not None:
                final_iterations, iteration_source = recommended, "current_validation"
            elif prior_recommended is not None:
                final_iterations, iteration_source = prior_recommended, "latest_prior_validation"
            else:
                final_iterations = {m.name: config.default_iterations for m in config.ensemble}
                iteration_source = "config_default"
            update_manifest(
                run_dir, stage="final_training_started",
                final_training_started_at_utc=utc_now_iso(),
                iteration_source=iteration_source,
            )
            output = fit_final_and_submit(
                train_features, test_features, sample, columns, config,
                run_dir, final_iterations,
            )
            result["submission"] = str(output)
            result["final_iterations"] = final_iterations
            result["iteration_source"] = iteration_source
            LOGGER.info("Submission written to %s", output)
        result["handoff_zip"] = str(paths.artifacts / f"{run_dir.name}_complete.zip")
        result_summary = {
            key: (value.to_dict(orient="records") if isinstance(value, pd.DataFrame) else value)
            for key, value in result.items()
        }
        write_json(run_dir / "result_summary.json", result_summary)
        update_manifest(
            run_dir, status="complete", stage="complete",
            completed_at_utc=utc_now_iso(), handoff_zip=result["handoff_zip"],
        )
        handoff_zip = create_handoff_zip(run_dir, "complete")
        result["handoff_zip"] = str(handoff_zip)
        return result
    except Exception:
        error_text = traceback.format_exc()
        (run_dir / "error_traceback.txt").write_text(error_text, encoding="utf-8")
        LOGGER.exception("Pipeline failed; preserving partial handoff")
        update_manifest(
            run_dir, status="failed", stage="failed",
            failed_at_utc=utc_now_iso(),
        )
        create_handoff_zip(run_dir, "failed")
        raise


## 11. Optimasi v3 — annual-memory ensemble dan OOF stacking

Versi ini menargetkan penurunan leaderboard RMSE dari baseline 1.597 menuju 0.81.
Angka 0.81 adalah **promotion target**, bukan klaim skor sebelum submission dinilai
oleh Kaggle karena label test tidak tersedia di notebook.

Perubahan utama:

1. **Annual target memory yang deployment-safe.** Test sepanjang 242 hari hanya
   memakai label 364–366 hari sebelumnya; seluruh timestamp sumber berada sebelum
   cutoff train. Median tiga lag menahan isolated sensor spike.
2. **Diversity objective.** CatBoost RMSE dan Huber serta LightGBM L2 dan Huber
   dilatih pada target robust-normalized yang sama.
3. **Perbaikan LightGBM Huber.** `alpha` memakai 0.90 (quantile parameter valid),
   bukan Huber delta CatBoost 1.5.
4. **OOF-weighted member blend.** Bobot lima learner dioptimalkan dengan kendala
   non-negatif dan jumlah satu, di-shrink ke equal weights, lalu diuji cross-fold.
5. **Representative folds.** Model selection memakai tiga fold yang sudah memiliki
   annual history. `sep_2024` mempunyai kalender horizon yang sama dengan test.
6. **OOF stacking.** Bobot blend model–annual-memory dipelajari dari OOF dan
   di-shrink per stasiun. Metrik cross-fold disimpan agar efek stacker tidak hanya
   dinilai secara in-sample.

Guard leakage memverifikasi bahwa horizon setiap fold lebih pendek daripada annual
lag minimum. Semua feature target-derived pada baris valid/test berasal dari waktu
sebelum cutoff; tidak ada recursive prediction.


In [ ]:
# ================================================================
# V3 OVERRIDES: safe annual memory + objective diversity + OOF stack
# ================================================================

ANNUAL_LAG_DAYS = (364, 365, 366, 728, 729, 730, 731)
V3_FOLDS = FOLDS[1:]  # folds with at least one annual cycle available
V3_STACKING_STATE: dict = {"global_weight": 0.0, "station_weights": {}}


def _assert_annual_lag_safety(folds=V3_FOLDS) -> None:
    """Prove that every annual source timestamp precedes its fold cutoff."""
    shortest_lag = min(ANNUAL_LAG_DAYS)
    for name, train_end, _valid_start, valid_end in folds:
        horizon = (pd.Timestamp(valid_end) - pd.Timestamp(train_end)).total_seconds() / 86_400
        if horizon >= shortest_lag:
            raise ValueError(
                f"Fold {name} horizon={horizon:.1f}d is not safe for "
                f"minimum annual lag={shortest_lag}d"
            )


def add_safe_annual_target_features(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target_history: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Attach only target values old enough to be known for the entire horizon.

    History is shifted forward by 364+ days and joined by exact station/timestamp.
    Consequently the value attached to row t is y(t-lag), never y(t) or a value
    inside the forecast horizon.
    """
    _assert_annual_lag_safety()
    history = target_history.loc[:, ["datetime", "nama_pos", "tma_mdpl"]].copy()
    history["datetime"] = pd.to_datetime(history["datetime"])

    def attach(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        for days in ANNUAL_LAG_DAYS:
            shifted = history.rename(columns={"tma_mdpl": f"annual_level_d{days}"}).copy()
            shifted["datetime"] = shifted["datetime"] + pd.Timedelta(days=days)
            out = out.merge(shifted, on=KEYS, how="left", validate="many_to_one")

        lag1 = [f"annual_level_d{d}" for d in (364, 365, 366)]
        lag2 = [f"annual_level_d{d}" for d in (728, 729, 730, 731)]
        out["annual_median_1y"] = out[lag1].median(axis=1, skipna=True)
        out["annual_mean_1y"] = out[lag1].mean(axis=1, skipna=True)
        out["annual_spread_1y"] = out[lag1].max(axis=1) - out[lag1].min(axis=1)
        out["annual_available_1y"] = out[lag1].notna().sum(axis=1).astype("int8")
        out["annual_median_2y"] = out[lag2].median(axis=1, skipna=True)
        out["annual_available_2y"] = out[lag2].notna().sum(axis=1).astype("int8")
        out["annual_yoy_delta"] = out["annual_median_1y"] - out["annual_median_2y"]
        out["annual_disagreement"] = (
            out["annual_median_1y"] - out["harmonic_baseline_mdpl"]
            if "harmonic_baseline_mdpl" in out else np.nan
        )
        return out

    return attach(train_features), attach(test_features)


# Wrap the audited feature builder instead of duplicating HydroRIVERS logic.
_build_feature_tables_v2 = build_feature_tables


def build_feature_tables(
    train: pd.DataFrame,
    test: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
    paths: Paths,
    config: ModelConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_features, test_features, weights = _build_feature_tables_v2(
        train, test, coordinates, environment, paths, config
    )
    train_features, test_features = add_safe_annual_target_features(
        train_features, test_features, train
    )
    return train_features, test_features, weights


# Recompute annual-vs-harmonic disagreement after the leakage-safe per-fold
# harmonic model has replaced the global NaN placeholders.
_apply_station_features_v2 = apply_station_features


def apply_station_features(
    fit_frame: pd.DataFrame,
    other_frames: Sequence[pd.DataFrame],
    config: ModelConfig,
) -> tuple[pd.DataFrame, list[pd.DataFrame]]:
    fit_out, others_out = _apply_station_features_v2(fit_frame, other_frames, config)
    frames = [fit_out, *others_out]
    for frame in frames:
        if "annual_median_1y" in frame and "harmonic_baseline_mdpl" in frame:
            frame["annual_disagreement"] = (
                frame["annual_median_1y"] - frame["harmonic_baseline_mdpl"]
            )
    return frames[0], frames[1:]


# Heterogeneous objectives. Names deliberately encode the objective so the
# existing immutable EnsembleMemberConfig does not need a schema migration.
V3_ENSEMBLE = (
    EnsembleMemberConfig("cb_rmse_d7", "catboost", seed=17, depth=7,
                         learning_rate=0.030, l2_leaf_reg=12.0),
    EnsembleMemberConfig("cb_huber_d6", "catboost", seed=41, depth=6,
                         learning_rate=0.040, l2_leaf_reg=10.0),
    EnsembleMemberConfig("cb_huber_d8", "catboost", seed=83, depth=8,
                         learning_rate=0.025, l2_leaf_reg=16.0),
    EnsembleMemberConfig("lgbm_l2", "lightgbm", seed=7, num_leaves=63,
                         learning_rate=0.025, feature_fraction=0.82,
                         bagging_fraction=0.82, min_child_samples=35),
    EnsembleMemberConfig("lgbm_huber", "lightgbm", seed=29, num_leaves=95,
                         learning_rate=0.020, feature_fraction=0.75,
                         bagging_fraction=0.82, min_child_samples=40),
)
V3_MEMBER_WEIGHTS = {member.name: 1.0 / len(V3_ENSEMBLE) for member in V3_ENSEMBLE}


def make_catboost_model(
    member: EnsembleMemberConfig,
    config: ModelConfig,
    iterations: int,
    train_dir: Path | None = None,
):
    from catboost import CatBoostRegressor

    is_rmse = "rmse" in member.name
    params = dict(
        iterations=iterations,
        learning_rate=member.learning_rate,
        depth=member.depth,
        loss_function="RMSE" if is_rmse else f"Huber:delta={config.huber_delta}",
        eval_metric="RMSE",  # promotion metric and early stopping are aligned
        l2_leaf_reg=member.l2_leaf_reg,
        random_seed=member.seed,
        random_strength=0.7,
        bootstrap_type="Bayesian",
        bagging_temperature=0.6,
        allow_writing_files=train_dir is not None,
        thread_count=-1,
        verbose=config.verbose_every,
    )
    if train_dir is not None:
        train_dir.mkdir(parents=True, exist_ok=True)
        params.update(train_dir=str(train_dir), save_snapshot=True,
                      snapshot_file="training.snapshot",
                      snapshot_interval=config.snapshot_interval_seconds)
    return CatBoostRegressor(**params)


def make_lightgbm_model(
    member: EnsembleMemberConfig,
    config: ModelConfig,
    iterations: int,
):
    import lightgbm as lgb

    is_l2 = "l2" in member.name
    return lgb.LGBMRegressor(
        objective="regression_l2" if is_l2 else "huber",
        alpha=0.90,  # valid LightGBM Huber/quantile alpha
        n_estimators=iterations,
        learning_rate=member.learning_rate,
        num_leaves=member.num_leaves,
        max_depth=-1,
        feature_fraction=member.feature_fraction,
        bagging_fraction=member.bagging_fraction,
        bagging_freq=1,
        min_child_samples=member.min_child_samples,
        reg_lambda=20.0,
        reg_alpha=0.10,
        random_state=member.seed,
        n_jobs=-1,
        verbosity=-1,
    )


def _read_saved_frame(stem: Path) -> pd.DataFrame:
    candidates = (stem.with_suffix(".parquet"), stem.with_suffix(".csv.gz"), stem.with_suffix(".csv"))
    for path in candidates:
        if path.exists():
            return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path, parse_dates=["datetime"])
    raise FileNotFoundError(f"No saved frame found for {stem}")


def _fit_annual_blend_weights(oof: pd.DataFrame) -> dict:
    """Fit conservative model/annual weights on level-1 OOF predictions."""
    usable = oof.dropna(subset=["prediction", "annual_median_1y", "tma_mdpl"]).copy()
    grid = np.linspace(0.0, 0.60, 31)
    if usable.empty:
        return {"global_weight": 0.0, "station_weights": {}, "rows": 0}

    def best_weight(frame: pd.DataFrame) -> tuple[float, float, float]:
        y = frame["tma_mdpl"].to_numpy(dtype=float)
        model = frame["prediction"].to_numpy(dtype=float)
        annual = frame["annual_median_1y"].to_numpy(dtype=float)
        base_rmse = float(np.sqrt(np.mean((y - model) ** 2)))
        scores = [float(np.sqrt(np.mean((y - ((1-w)*model + w*annual)) ** 2))) for w in grid]
        pos = int(np.argmin(scores))
        return float(grid[pos]), base_rmse, scores[pos]

    global_w, base_rmse, blend_rmse = best_weight(usable)
    station_weights = {}
    for station, part in usable.groupby("nama_pos", observed=True):
        local_w, local_base, local_blend = best_weight(part)
        # Do not accept microscopic in-sample improvements. Shrink local weights
        # toward the global solution because each station has only a few folds.
        if local_base - local_blend < 0.005:
            local_w = global_w
        shrink = len(part) / (len(part) + 1500.0)
        station_weights[str(station)] = float(global_w + shrink * (local_w - global_w))
    return {
        "global_weight": global_w,
        "station_weights": station_weights,
        "rows": int(len(usable)),
        "model_rmse": base_rmse,
        "in_sample_stacked_rmse": blend_rmse,
    }


def _fit_member_weights(frame: pd.DataFrame, member_names: Sequence[str]) -> dict[str, float]:
    """Constrained RMSE-optimal blend with mild shrinkage toward equal weights."""
    from scipy.optimize import minimize

    cols = [f"prediction_{name}" for name in member_names]
    usable = frame.dropna(subset=["tma_mdpl", *cols])
    equal = np.full(len(member_names), 1.0 / len(member_names), dtype=float)
    if usable.empty or len(member_names) == 1:
        return {name: float(w) for name, w in zip(member_names, equal)}
    matrix = usable[cols].to_numpy(dtype=float)
    target = usable["tma_mdpl"].to_numpy(dtype=float)
    result = minimize(
        lambda w: np.mean((target - matrix @ w) ** 2),
        x0=equal,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * len(member_names),
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        options={"maxiter": 500, "ftol": 1e-12},
    )
    optimum = result.x if result.success else equal
    # Ten-percent shrinkage reduces fold-to-fold weight instability.
    stable = 0.90 * optimum + 0.10 * equal
    stable /= stable.sum()
    return {name: float(w) for name, w in zip(member_names, stable)}


def _apply_member_weights(
    frame: pd.DataFrame, member_names: Sequence[str], weights: dict[str, float]
) -> np.ndarray:
    matrix = frame[[f"prediction_{name}" for name in member_names]].to_numpy(dtype=float)
    vector = np.array([weights.get(name, 0.0) for name in member_names], dtype=float)
    if vector.sum() <= 0:
        vector[:] = 1.0 / len(vector)
    else:
        vector /= vector.sum()
    return matrix @ vector


def _apply_annual_blend(frame: pd.DataFrame, state: dict) -> np.ndarray:
    weights = frame["nama_pos"].map(state.get("station_weights", {})).fillna(
        state.get("global_weight", 0.0)
    ).to_numpy(dtype=float, copy=True)
    annual = frame["annual_median_1y"].to_numpy(dtype=float)
    model = frame["prediction"].to_numpy(dtype=float)
    weights[~np.isfinite(annual)] = 0.0
    return (1.0 - weights) * model + weights * np.nan_to_num(annual, nan=0.0)


_run_validation_v2 = run_validation


def run_validation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds=V3_FOLDS,
    members=None,
    write_artifacts: bool = True,
):
    """Run audited validation, then fit and cross-check the annual stacker."""
    global V3_STACKING_STATE, V3_MEMBER_WEIGHTS
    _assert_annual_lag_safety(folds)
    report, recommended = _run_validation_v2(
        train_features, columns, config, artifacts,
        folds=folds, members=members, write_artifacts=write_artifacts,
    )
    if not write_artifacts:
        return report, recommended

    oof = _read_saved_frame(artifacts / "metrics" / "oof_predictions")
    annual_cols = ["datetime", "nama_pos", "annual_median_1y"]
    oof["datetime"] = pd.to_datetime(oof["datetime"])
    # train_features may have duplicate (datetime, nama_pos) rows if the feature
    # builder produced repeated index entries; deduplicate before merging so the
    # one_to_one contract holds.
    annual_lookup = (
        train_features[annual_cols]
        .drop_duplicates(subset=["datetime", "nama_pos"], keep="first")
    )
    oof = oof.merge(annual_lookup, on=["datetime", "nama_pos"],
                    how="left", validate="one_to_one")

    member_names = [m.name for m in (members or config.ensemble)]
    missing_member_cols = [f"prediction_{name}" for name in member_names if f"prediction_{name}" not in oof]
    if missing_member_cols:
        raise RuntimeError(f"Missing OOF member predictions: {missing_member_cols}")

    # Honest cross-fold evaluation: both member weights and annual blend weights
    # for a held fold are learned only from the other folds.
    crossfit = pd.Series(index=oof.index, dtype=float)
    crossfit_rows = []
    for held_fold, held in oof.groupby("fold", observed=True):
        level1_train = oof[oof["fold"] != held_fold].copy()
        if level1_train.empty:
            member_state = {name: 1.0 / len(member_names) for name in member_names}
        else:
            member_state = _fit_member_weights(level1_train, member_names)
        level1_train["prediction"] = _apply_member_weights(level1_train, member_names, member_state)
        annual_state = _fit_annual_blend_weights(level1_train)
        held_eval = held.copy()
        held_eval["prediction"] = _apply_member_weights(held_eval, member_names, member_state)
        pred = _apply_annual_blend(held_eval, annual_state)
        crossfit.loc[held.index] = pred
        y = held["tma_mdpl"].to_numpy(dtype=float)
        crossfit_rows.append({
            "fold": held_fold,
            "stacked_crossfit_rmse": float(np.sqrt(np.mean((y-pred)**2))),
            "stacked_crossfit_mae": float(np.mean(np.abs(y-pred))),
        })
    oof["stacked_crossfit_prediction"] = crossfit
    save_frame(oof, artifacts / "metrics" / "oof_predictions_v3")
    crossfit_table = pd.DataFrame(crossfit_rows)
    crossfit_table.to_csv(artifacts / "metrics" / "stacking_crossfit_metrics.csv", index=False)
    report = report.merge(crossfit_table, on="fold", how="left", validate="one_to_one")
    report.to_csv(artifacts / "metrics" / "validation_metrics_v3.csv", index=False)

    V3_MEMBER_WEIGHTS = _fit_member_weights(oof, member_names)
    oof["equal_weight_prediction"] = oof["prediction"]
    oof["prediction"] = _apply_member_weights(oof, member_names, V3_MEMBER_WEIGHTS)
    V3_STACKING_STATE = _fit_annual_blend_weights(oof)
    write_json(artifacts / "metrics" / "member_weights.json", V3_MEMBER_WEIGHTS)
    write_json(artifacts / "metrics" / "stacking_weights.json", V3_STACKING_STATE)
    LOGGER.info("V3 member weights: %s", V3_MEMBER_WEIGHTS)
    LOGGER.info("V3 annual stacking state: %s", V3_STACKING_STATE)
    return report, recommended


_fit_final_and_submit_v2 = fit_final_and_submit


def fit_final_and_submit(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    sample: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    iterations_by_member: dict[str, int],
) -> Path:
    """Fit the full ensemble, then apply OOF-learned conservative stacking."""
    output = _fit_final_and_submit_v2(
        train_features, test_features, sample, columns, config,
        artifacts, iterations_by_member,
    )
    submission = pd.read_csv(output)
    model_only = artifacts / "submission_model_only.csv"
    submission.to_csv(model_only, index=False)

    keys = test_features[["id", "nama_pos", "annual_median_1y"]].copy()
    stacked = submission.merge(keys, on="id", how="left", validate="one_to_one")
    stacked = stacked.rename(columns={"tma_mdpl": "prediction"})
    stacked["tma_mdpl"] = _apply_annual_blend(stacked, V3_STACKING_STATE)
    if not np.isfinite(stacked["tma_mdpl"]).all():
        raise ValueError("V3 stacked submission contains non-finite predictions")
    stacked[["id", "tma_mdpl"]].to_csv(output, index=False)
    write_json(artifacts / "v3_postprocess.json", {
        "stacking_state": V3_STACKING_STATE,
        "model_only_submission": str(model_only),
        "final_submission": str(output),
    })
    return output


## 11. Konfigurasi mode dan toggle optimasi

- `audit`: validasi input saja, tanpa training.
- `tune`: sweep hyperparameter CatBoost ringan pada satu fold (bagian 8c).
- `validate`: preprocessing dan rolling-origin validation ensemble penuh, tanpa final fit.
- `ablation`: on/off test spike-downweighting dan feature pruning (bagian 8b).
- `fit`: final fit ensemble dan submission, memakai iterasi per member yang direkomendasikan.
- `all`: validasi, final fit, submission, dan ZIP handoff lengkap.

Pada Kaggle, input otomatis memakai `/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026` dan seluruh output memakai `/kaggle/working`.

Toggle `USE_SPIKE_WEIGHTS`, `PRUNE_FEATURES`, `FEATURE_IMPORTANCE_SOURCE`, dan `ENABLE_QUANTILE_MODEL` di cell berikut mengontrol optimasi yang diusulkan pada review sebelumnya — defaultnya aman (spike weighting aktif, pruning dan model quantile mati) sampai dibuktikan lewat mode `ablation`/`tune`.


In [ ]:
# Guard: Run All hanya melakukan audit dan TIDAK melatih model.
KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026")
ON_KAGGLE = KAGGLE_DATA_DIR.exists()

MODE = "all"  # pilihan: audit, ablation, tune, validate, fit, all
DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else Path("datasets")
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")

# --- Toggle hasil review senior (lihat markdown tiap bagian untuk alasannya) ---
USE_SPIKE_WEIGHTS = False           # matikan untuk ablasi poin 3
PRUNE_FEATURES = False              # aktifkan setelah FEATURE_IMPORTANCE_SOURCE tersedia
FEATURE_IMPORTANCE_SOURCE = '/kaggle/working/handoff_20260718T125208Z_validate/metrics/feature_importance.csv'   # contoh: OUTPUT_DIR / "handoff_..." / "metrics" / "feature_importance.csv"
PRUNE_KEEP_FRACTION = 0.85
ENABLE_QUANTILE_MODEL = False       # CatBoost MultiQuantile tambahan (poin 4)
ENABLE_HARMONIC_FEATURE = True      # fitur baseline harmonik per pos (follow-up Wonogiri Dam)
HARMONIC_RIDGE_ALPHA = 5.0          # regularisasi Ridge; naikkan kalau kurva per pos terlalu "liar"
ENABLE_ANCHOR_FEATURE = True        # horizon_days + recent_anomaly + anchor_basis (follow-up station-aware stacking tim lain)
ANCHOR_RECENT_WINDOW_DAYS = 30.0    # jendela hari terakhir sebelum cutoff buat hitung recent_anomaly per pos
ANCHOR_TAU_DAYS = 365.0             # skala peluruhan horizon; makin besar makin lama anomali "dipercaya"
ITERATIONS_BY_MEMBER = None         # None = pakai rekomendasi validasi/prior run per member
SAVE_FEATURE_MATRICES = True        # disarankan untuk handoff penyempurnaan

CONFIG = ModelConfig(
    ensemble=V3_ENSEMBLE,
    use_spike_weights=USE_SPIKE_WEIGHTS,
    prune_features=PRUNE_FEATURES,
    feature_importance_source=str(FEATURE_IMPORTANCE_SOURCE) if FEATURE_IMPORTANCE_SOURCE else None,
    prune_keep_fraction=PRUNE_KEEP_FRACTION,
    enable_quantile_model=ENABLE_QUANTILE_MODEL,
    enable_harmonic_feature=ENABLE_HARMONIC_FEATURE,
    harmonic_ridge_alpha=HARMONIC_RIDGE_ALPHA,
    enable_anchor_feature=ENABLE_ANCHOR_FEATURE,
    anchor_recent_window_days=ANCHOR_RECENT_WINDOW_DAYS,
    anchor_tau_days=ANCHOR_TAU_DAYS,
)

print(f"Runtime: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Input : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Ensemble members ({len(CONFIG.ensemble)}): {[m.name for m in CONFIG.ensemble]}")
print(f"use_spike_weights={CONFIG.use_spike_weights} prune_features={CONFIG.prune_features} "
      f"enable_quantile_model={CONFIG.enable_quantile_model} enable_harmonic_feature={CONFIG.enable_harmonic_feature} "
      f"enable_anchor_feature={CONFIG.enable_anchor_feature}")


## 12. Jalankan mode yang dipilih


In [ ]:
result = run_pipeline(
    mode=MODE,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
    iterations_by_member=ITERATIONS_BY_MEMBER,
    save_feature_matrices=SAVE_FEATURE_MATRICES,
)
result
